In [38]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql.types import NumericType, DoubleType
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import countDistinct, abs, stddev, col as spark_col
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import input_file_name, regexp_extract, lit, create_map, col, substring
from itertools import chain
import os
import tarfile
import glob
from datetime import date, datetime, timedelta

In [8]:
from pyspark.sql import SparkSession

try:
    print(spark.sparkContext.master)
    print(spark.sparkContext.isStopped())
except Exception as e:
    print(e)

spark://spark-master-svc:7077
'SparkContext' object has no attribute 'isStopped'


In [37]:
spark.stop()

In [39]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = SparkConf().setAll([

    ('spark.executor.memory', '22g'),
    ('spark.executor.cores', '3'),
    ('spark.driver.memory', '12g'),
    ('spark.driver.maxResultSize', '8g'),   # <-- Add this
    ('spark.cores.max', '6'),

    ("spark.ui.enabled", "true"),
    ("spark.port.maxRetries", "100"),

    ('spark.app.name', 'alarm_treatment'),

    ('spark.sql.files.ignoreCorruptFiles', 'true'),

    ('spark.master', 'spark://spark-master-svc:7077')

])

spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [41]:
spark

In [4]:
# Check your spark context status
print(spark.sparkContext.master)
print(spark.sparkContext.uiWebUrl)

spark://spark-master-7c47f4dd8f-btxw8:7077
http://10.17.10.20:4041


In [106]:
base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/kpi_data"

kpi_1month_df = (
    spark.read
    .option("basePath", base_path)
    .parquet(base_path)
    .filter(
        col("DATE_ID").between("2026-06-23", "2026-07-22")
    )
)

Py4JJavaError: An error occurred while calling o1821.parquet.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection

In [107]:
kpi_1month_df.printSchema()

root
 |-- ENodeBFunction: string (nullable = true)
 |-- COLLINT: string (nullable = true)
 |-- processed_time: string (nullable = true)
 |-- SITE: string (nullable = true)
 |-- CELLID: string (nullable = true)
 |-- COLLECTTIME: string (nullable = true)
 |-- LTE_RRC_Setup_SR: double (nullable = true)
 |-- LTE_ERAB_Drop_Rate: double (nullable = true)
 |-- LTE_DL_Cell_Thpt_Mbps: double (nullable = true)
 |-- LTE_UL_User_Thpt_Mbps: double (nullable = true)
 |-- LTE_DL_User_Thpt_Mbps: double (nullable = true)
 |-- LTE_DL_User_Thpt_Avg_DistrMbps: double (nullable = true)
 |-- LTE_DL_User_Thpt_Avg_Filtered_DistrMbps: double (nullable = true)
 |-- LTE_DL_CA_User_Thpt_Mbps: double (nullable = true)
 |-- LTE_DL_BLER: double (nullable = true)
 |-- LTE_DL_BLER_QPSK: double (nullable = true)
 |-- LTE_DL_BLER_16QAM: double (nullable = true)
 |-- LTE_DL_BLER_64QAM: double (nullable = true)
 |-- LTE_DL_BLER_256QAM: double (nullable = true)
 |-- LTE_DL_Modulation_Rate_QPSK: double (nullable = true)
 |-

In [12]:
base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/NRCell"

kpi_1month_5G_df = (
    spark.read
    .option("basePath", base_path)
    .parquet(base_path)
    .filter(
        col("DATE_ID").between("2026-06-23", "2026-07-22")
    )
)

In [13]:
kpi_1month_5G_df.count()

78246849

In [14]:
kpi_1month_5G_df.printSchema()

root
 |-- COLLECTTIME: string (nullable = true)
 |-- CELLID: string (nullable = true)
 |-- SITE: string (nullable = true)
 |-- NR_SA_Paging_Discard_Rate: double (nullable = true)
 |-- NR_DL_Active_UEs: double (nullable = true)
 |-- NR_UL_Active_UEs: double (nullable = true)
 |-- NR_DL_MAC_Volume_MB: double (nullable = true)
 |-- NR_UL_MAC_Volume_MB: double (nullable = true)
 |-- NR_DL_Active_UEs_True: double (nullable = true)
 |-- NR_UL_Active_UEs_True: double (nullable = true)
 |-- NR_DL_RBSym_Util: double (nullable = true)
 |-- NR_UL_RBSym_Util: double (nullable = true)
 |-- NR_Msg2_Attempt_SR: double (nullable = true)
 |-- NR_Msg2_Attempts: double (nullable = true)
 |-- NR_RACH_SR: double (nullable = true)
 |-- NR_RACH_Att: double (nullable = true)
 |-- NR_UL_MAC_Cell_Thp_total_time_Mbps: double (nullable = true)
 |-- NR_DL_MAC_Cell_Thp_total_time_Mbps: double (nullable = true)
 |-- NR_DL_HARQ_BLER_256QAM: double (nullable = true)
 |-- NR_DL_RLC_BLER: double (nullable = true)
 |-- N

In [9]:
kpi_1month_5G_df.groupBy("SITE").agg(countDistinct("CELLID").alias("cell_count")).orderBy("cell_count", ascending=False).show(truncate=False)

+--------------+----------+
|SITE          |cell_count|
+--------------+----------+
|BKFTS02L_6NB04|9         |
|CMI7012T_2NB02|8         |
|CMI0128T_2NB02|8         |
|CMI6188F_ANB04|8         |
|CMI0922T_2NB02|8         |
|CMI1742T_2NB04|8         |
|CMI6841T_2NB02|8         |
|CRI6262L_6NB02|6         |
|LBR7155L_6NB02|6         |
|NKT7400L_6NB03|6         |
|CRI2009L_6NB02|6         |
|CRI6315L_6NB02|6         |
|CMI2116L_6NB02|6         |
|NKT7414L_6NB06|6         |
|KPP1616L_6NB02|6         |
|CRI6273L_6NB02|6         |
|CMI7111T_2NB01|6         |
|NKW1915L_6NB02|6         |
|LPN1782L_6NB02|6         |
|PCB7292L_6NB02|6         |
+--------------+----------+
only showing top 20 rows



In [10]:
selected_site = "AYT0381P_9NB01"

site_df = (
    kpi_1month_5G_df
    .filter(F.col("SITE") == selected_site)
    .orderBy("COLLECTTIME")
)

site_df.select(
    "SITE",
    "CELLID",
    "COLLECTTIME",
    "NR_Cell_Availability"
).show(100, truncate=False)

+--------------+------------------+--------------+--------------------+
|SITE          |CELLID            |COLLECTTIME   |NR_Cell_Availability|
+--------------+------------------+--------------+--------------------+
|AYT0381P_9NB01|AYT0381H_7NB01_S02|20260623001500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S03|20260623001500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S01|20260623001500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623001500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S02|20260623003000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S03|20260623003000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623003000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S01|20260623003000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623004500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S02|20260623004500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S03|20260623004500|100.0         

In [11]:
# Find cells that have gone to outage (NR_Cell_Availability == 0)
outage_cells = (
    site_df
    .filter(F.col("NR_Cell_Availability") == 0)
    .select("CELLID")
    .distinct()
)

outage_cells.show(truncate=False)


+------------------+
|CELLID            |
+------------------+
|AYT0381H_7NB01_S03|
|AYT0381H_7NB01_S01|
|AYT0381H_7NB01_S04|
|AYT0381H_7NB01_S02|
+------------------+



In [12]:
cell_id = "AYT0381H_7NB01_S04"

cell_df = (
    site_df
    .filter(F.col("CELLID") == cell_id)
    .orderBy("COLLECTTIME")
)

cell_df.select(
    "SITE",
    "CELLID",
    "COLLECTTIME",
    "NR_Cell_Availability"
    ).show(200, truncate=False)

+--------------+------------------+--------------+--------------------+
|SITE          |CELLID            |COLLECTTIME   |NR_Cell_Availability|
+--------------+------------------+--------------+--------------------+
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623001500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623003000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623004500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623010000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623011500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623013000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623014500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623020000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623021500|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623023000|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260623024500|100.0         

In [13]:
# ============================================================
# PRE-OUTAGE KPI ANALYSIS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio


# ============================================================
# CONFIGURATION
# ============================================================

selected_site = "AYT0381P_9NB01"
cell_id = "AYT0381H_7NB01_S04"

target = "NR_Cell_Availability"

# How much history you want before outage
HOURS_BEFORE_OUTAGE = 96


# ============================================================
# 1. FILTER SITE + CELL
# ============================================================

cell_df = (
    kpi_1month_5G_df
    .filter(
        (F.col("SITE") == selected_site)
        &
        (F.col("CELLID") == cell_id)
    )
)


# ============================================================
# 2. CONVERT COLLECTTIME
# ============================================================

# Your COLLECTTIME:
# 20260701143000
#
# Format:
# yyyyMMddHHmmss

cell_df = cell_df.withColumn(
    "COLLECTTIME_TS",
    F.to_timestamp(
        F.col("COLLECTTIME").cast("string"),
        "yyyyMMddHHmmss"
    )
)


# ============================================================
# 3. SORT CELL DATA
# ============================================================

cell_window = (
    Window
    .partitionBy(
        "SITE",
        "CELLID"
    )
    .orderBy(
        "COLLECTTIME_TS"
    )
)


# ============================================================
# 4. GET PREVIOUS AVAILABILITY
# ============================================================

cell_df = cell_df.withColumn(
    "previous_availability",
    F.lag(
        F.col(target)
    ).over(cell_window)
)


# ============================================================
# 5. IDENTIFY OUTAGE START
# ============================================================
#
# We DON'T want:
#
# 14:30 -> 0  <- outage starts
# 14:45 -> 0  <- same outage
#
# We only want 14:30.
#
# Therefore:
#
# current availability = 0
# previous availability > 0
#
# ============================================================

cell_df = cell_df.withColumn(
    "outage_start",
    F.when(
        (F.col(target) == 0)
        &
        (F.col("previous_availability") > 0),
        1
    ).otherwise(0)
)


# ============================================================
# 6. SEE ALL OUTAGE STARTS
# ============================================================

print("Outage starts:")

(
    cell_df
    .filter(
        F.col("outage_start") == 1
    )
    .select(
        "SITE",
        "CELLID",
        "COLLECTTIME",
        "COLLECTTIME_TS",
        "previous_availability",
        target
    )
    .orderBy("COLLECTTIME_TS")
    .show(
        100,
        truncate=False
    )
)

Outage starts:


+--------------+------------------+--------------+-------------------+---------------------+--------------------+
|SITE          |CELLID            |COLLECTTIME   |COLLECTTIME_TS     |previous_availability|NR_Cell_Availability|
+--------------+------------------+--------------+-------------------+---------------------+--------------------+
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260624123000|2026-06-24 12:30:00|100.0                |0.0                 |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260701100000|2026-07-01 10:00:00|100.0                |0.0                 |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260701143000|2026-07-01 14:30:00|61.444               |0.0                 |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|20260721150000|2026-07-21 15:00:00|100.0                |0.0                 |
+--------------+------------------+--------------+-------------------+---------------------+--------------------+



In [14]:
# ============================================================
# 7. SELECT OUTAGE
# ============================================================

outage_rows = (
    cell_df
    .filter(
        F.col("outage_start") == 1
    )
    .orderBy("COLLECTTIME_TS")
#     .first()
    .limit(2)
    .collect()
)


if len(outage_rows) < 2:
    raise ValueError(
        f"Less than 2 healthy -> outage transitions found for {cell_id}"
    )
    
outage_row = outage_rows[1]


outage_time = outage_row["COLLECTTIME_TS"]

print(
    "Selected outage time:",
    outage_time
)

26/07/27 12:50:03 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Selected outage time: 2026-07-01 10:00:00


In [15]:
# ============================================================
# 8. TAKE ONLY PRE-OUTAGE DATA
# ============================================================

pre_outage_df = (
    cell_df
    .filter(

        (
            F.col("COLLECTTIME_TS")
            >=
            F.expr(
                f"timestamp'{outage_time}' - INTERVAL {HOURS_BEFORE_OUTAGE} HOURS"
            )
        )

        &

        (
            F.col("COLLECTTIME_TS")
            <=
            F.lit(outage_time)
        )
    )
    .orderBy(
        "COLLECTTIME_TS"
    )
)

In [16]:
pre_outage_df.select(
    "SITE",
    "CELLID",
    "COLLECTTIME_TS",
    target
).show(
    100,
    truncate=False
)

+--------------+------------------+-------------------+--------------------+
|SITE          |CELLID            |COLLECTTIME_TS     |NR_Cell_Availability|
+--------------+------------------+-------------------+--------------------+
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 10:00:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 10:15:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 10:30:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 10:45:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 11:00:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 11:15:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 11:30:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 11:45:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 12:00:00|100.0               |
|AYT0381P_9NB01|AYT0381H_7NB01_S04|2026-06-27 12:15:00|100.0               |

In [17]:
target = "NR_Cell_Availability"


# ============================================================
# 2. AUTOMATICALLY GET ALL NUMERIC KPI COLUMNS
# ============================================================

exclude_cols = {
    "SITE",
    "CELLID",
    "COLLECTTIME",
    target
}

kpis_to_plot = [
    field.name
    for field in site_df.schema.fields
    if isinstance(field.dataType, NumericType)
    and field.name not in exclude_cols
]

In [18]:
# ============================================================
# 9. CONVERT PRE-OUTAGE DATA TO PANDAS
# ============================================================

columns_needed = (
    [
        "SITE",
        "CELLID",
        "COLLECTTIME_TS",
        target
    ]
    +
    kpis_to_plot
)


cell_pd = (
    pre_outage_df
    .select(*columns_needed)
    .toPandas()
)

cell_pd["COLLECTTIME_TS"] = pd.to_datetime(
    cell_pd["COLLECTTIME_TS"]
)

cell_pd = cell_pd.sort_values(
    "COLLECTTIME_TS"
)


print(cell_pd[
    [
        "COLLECTTIME_TS",
        target
    ]
])

         COLLECTTIME_TS  NR_Cell_Availability
0   2026-06-27 10:00:00                 100.0
1   2026-06-27 10:15:00                 100.0
2   2026-06-27 10:30:00                 100.0
3   2026-06-27 10:45:00                 100.0
4   2026-06-27 11:00:00                 100.0
..                  ...                   ...
318 2026-06-30 22:30:00                 100.0
319 2026-06-30 22:45:00                 100.0
320 2026-06-30 23:15:00                 100.0
321 2026-06-30 23:30:00                 100.0
322 2026-07-01 10:00:00                   0.0

[323 rows x 2 columns]


/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


In [19]:
from pyspark.sql.functions import col, sum as spark_sum

kpi_1month_5G_df.select(spark_sum(col("COLLECTTIME").isNull().cast("int")).alias("COLLECTTIME_nulls")).show()


+-----------------+
|COLLECTTIME_nulls|
+-----------------+
|                0|
+-----------------+



In [20]:
(
    kpi_1month_5G_df
    .filter(
        (F.col("SITE") == "AYT0381P_9NB01") &
        (F.col("CELLID") == "AYT0381H_7NB01_S01") &
        (F.col("COLLECTTIME").cast("string") == "20260630100000")
    )
    .select(
        "SITE",
        "CELLID",
        "COLLECTTIME",
        "NR_Msg2_Attempt_SR"
    )
    .show(truncate=False)
)

+--------------+------------------+--------------+------------------+
|SITE          |CELLID            |COLLECTTIME   |NR_Msg2_Attempt_SR|
+--------------+------------------+--------------+------------------+
|AYT0381P_9NB01|AYT0381H_7NB01_S01|20260630100000|94.41             |
+--------------+------------------+--------------+------------------+



In [21]:
kpi_1month_5G_df.select("essEnabled").show(10)

+----------+
|essEnabled|
+----------+
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
|         0|
|         0|
|         0|
|         1|
+----------+
only showing top 10 rows



In [22]:
import plotly.graph_objects as go
import plotly.io as pio

figures = []

# kpis_to_plot1 = [
#     col for col in cell_pd.columns
#     if col not in exclude_cols
#     and pd.api.types.is_numeric_dtype(cell_pd[col])
# ]

kpis_to_plot1 = [
    "NR_DL_Active_UEs",
    "NR_UL_Active_UEs",
    "NR_DL_MAC_Volume_MB",
    "NR_UL_MAC_Volume_MB",
    "NR_DL_RBSym_Util",
    "NR_RACH_Att",
    "NR_Avg_DL_MAC_Thp_Mbps",
    "NR_Avg_UL_MAC_UE_Thp_Mbps",
    "NR_DL_HARQ_BLER",
    "NR_DL_DTX_Rate",
    "NR_Avg_UL_PUCCH_SINR_dB",
    "NR_Avg_UL_PUSCH_SINR_dB",
    "NR_Avg_CQI",
    "NR_Avg_UL_Pathloss_dB",
    "NR_UL_RSSI",
    "NR_TA",
    "NR_Avg_DL_Latency_ms",
    "NR_ENDC_Setup_Att",
    "NR_ENDC_Setup_SR",
    "NR_SA_RRC_ATT",
    "NR_SgNB_Abnormal_RR",
    "ESS_TOTAL_PRB_AVG_UTIL_DL",
    "ESS_NR_PRB_AVG_UTIL_DL",
    "NR_DL_Active_UEs_True",
    "NR_Msg2_Attempt_SR"
]

#     "NR_DL_RBSym_Util",
#     "ESS_NR_PRB_AVG_UTIL_DL",
#     "essEnabled"

for kpi in kpis_to_plot1:

    fig = go.Figure()


    # ========================================================
    # KPI
    # ========================================================

    fig.add_trace(
        go.Scatter(
            x=cell_pd["COLLECTTIME_TS"],
            y=cell_pd[kpi],

            mode="lines+markers",

            name=kpi
        )
    )


    # ========================================================
    # TARGET - NR CELL AVAILABILITY
    # ========================================================

    fig.add_trace(
        go.Scatter(
            x=cell_pd["COLLECTTIME_TS"],
            y=cell_pd[target],

            mode="lines+markers",

            name=target,

            yaxis="y2"
        )
    )


    # ========================================================
    # OUTAGE START VERTICAL LINE
    # ========================================================

    fig.add_shape(
        type="line",

        x0=outage_time,
        x1=outage_time,

        y0=0,
        y1=1,

        xref="x",
        yref="paper",

        line=dict(
            dash="dash",
            width=2
        )
    )


    # ========================================================
    # OUTAGE START ANNOTATION
    # ========================================================

    fig.add_annotation(
        x=outage_time,
        y=1,

        xref="x",
        yref="paper",

        text="Outage Start",

        showarrow=True,

        arrowhead=2,

        yshift=15
    )


    # ========================================================
    # LAYOUT
    # ========================================================

    fig.update_layout(

        title=(
            f"{kpi} - {HOURS_BEFORE_OUTAGE} Hours "
            f"Before Outage"
        ),

        xaxis=dict(
            title="Time"
        ),

        yaxis=dict(
            title=kpi
        ),

        yaxis2=dict(
            title="NR Cell Availability",

            overlaying="y",

            side="right",

            range=[0, 105]
        ),

        hovermode="x unified",

        height=500
    )


    figures.append(fig)

In [23]:
# ============================================================
# 11. CREATE HTML REPORT
# ============================================================

html_content = f"""
<html>

<head>
    <title>Pre-Outage KPI Analysis - {cell_id}</title>
</head>

<body>

<h1>Pre-Outage KPI Trend Analysis</h1>

<h2>Site: {selected_site}</h2>

<h2>Cell: {cell_id}</h2>

<p>
Outage Start: {outage_time}
</p>

<p>
Analysis Window: {HOURS_BEFORE_OUTAGE} hours before outage
</p>

<p>
Each graph shows KPI behaviour leading up to the outage.
The graph ends when NR_Cell_Availability reaches 0.
</p>

<hr>
"""


for i, fig in enumerate(figures):

    html_content += pio.to_html(
        fig,

        full_html=False,

        include_plotlyjs=(
            True if i == 0 else False
        )
    )

    html_content += "<hr>"


html_content += """

</body>

</html>
"""


file_name = (
    f"Pre_Outage_KPI_Analysis_{cell_id} selected KPIs.html"
)


with open(
    file_name,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html_content
    )


print(
    "Report created:",
    file_name
)

Report created: Pre_Outage_KPI_Analysis_AYT0381H_7NB01_S04 selected KPIs.html


In [24]:
# Target KPI
target = "NR_Cell_Availability"

# Select numeric KPI columns
kpis_to_plot1 = [
    col for col in cell_pd.columns
    if col not in exclude_cols
    and pd.api.types.is_numeric_dtype(cell_pd[col])
    and col != target
]

# Calculate correlation
correlations = (
    cell_pd[kpis_to_plot1]
    .corrwith(cell_pd[target])
    .dropna()
)

# Create DataFrame
correlation_df = correlations.reset_index()
correlation_df.columns = ["KPI", "Correlation"]

# Add absolute correlation
correlation_df["Abs_Correlation"] = correlation_df["Correlation"].abs()

# Sort strongest correlations first
correlation_df = (
    correlation_df
    .sort_values("Abs_Correlation", ascending=False)
    .reset_index(drop=True)
)

correlation_df

,KPI,Correlation,Abs_Correlation
0,NR_Cell_Availability_Num,-1.000000,1.000000
1,NR_Msg2_Attempt_SR,0.997253,0.997253
2,NR_Avg_UL_Pathloss_95percentile_dB,0.662564,0.662564
3,NR_Avg_CQI,0.402129,0.402129
4,NR_DL_HARQ_BLER_256QAM,-0.261238,0.261238
5,NR_Avg_CQI_256QAM,0.179881,0.179881
6,NR_ENDC_RRC_CU_Max,0.144517,0.144517
7,NR_ENDC_Setup_Att,0.119502,0.119502
8,NR_Msg2_Attempts,0.106481,0.106481
9,NR_RACH_Att,0.106065,0.106065


In [25]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
import pandas as pd

# ============================================================
# CORRELATION OF NR_Cell_Availability WITH ALL NUMERIC KPIs
# ============================================================

target = "NR_Cell_Availability"

# 1. Check target exists
if target not in kpi_1month_5G_df.columns:
    raise ValueError(f"{target} not found in DataFrame")

# 2. Find all numeric columns
numeric_cols = [
    field.name
    for field in kpi_1month_5G_df.schema.fields
    if isinstance(field.dataType, NumericType)
]

# 3. Remove target itself
kpi_cols = [
    col for col in numeric_cols
    if col != target
]

print("Number of numeric KPIs:", len(kpi_cols))

# 4. Calculate correlations in ONE Spark aggregation
corr_exprs = [
    F.corr(F.col(target), F.col(kpi)).alias(kpi)
    for kpi in kpi_cols
]

corr_row = (
    kpi_1month_5G_df
    .select(target, *kpi_cols)
    .agg(*corr_exprs)
    .first()
)

# 5. Convert only correlation results to Pandas
correlation_df = pd.DataFrame({
    "KPI": kpi_cols,
    "Correlation": [corr_row[kpi] for kpi in kpi_cols]
})

# 6. Remove KPIs where correlation couldn't be calculated
correlation_df = correlation_df.dropna(
    subset=["Correlation"]
)

# 7. Absolute correlation for ranking
correlation_df["Abs_Correlation"] = (
    correlation_df["Correlation"].abs()
)

# 8. Strongest correlations first
correlation_df = (
    correlation_df
    .sort_values(
        "Abs_Correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

correlation_df.head(30)

Number of numeric KPIs: 80


,KPI,Correlation,Abs_Correlation
0,NR_Cell_Availability_Num,-1.000000,1.000000
1,NR_Avg_UL_Pathloss_95percentile_dB,0.232394,0.232394
2,NR_Avg_CQI,0.199601,0.199601
3,NR_SgNB_Abnormal_RR,-0.119548,0.119548
4,ESS_TOTAL_PRB_AVG_UTIL_DL,0.032561,0.032561
5,NR_ENDC_Setup_SR,0.031040,0.031040
6,NR_ENDC_RRC_CU_Max,0.029299,0.029299
7,NR_ENDC_Setup_Att,0.029248,0.029248
8,ESS_NR_PRB_AVG_UTIL_DL,0.026433,0.026433
9,NR_ENDC_SARR,-0.026259,0.026259


In [47]:
itsm_base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/itsm_hpd_tms_data_hourly"

ticket_1month_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(f"{itsm_base_path}/*/*/*")
    .withColumn("folder_date", regexp_extract(col("_metadata.file_path"), r"(\d{4}-\d{2}-\d{2})", 1))
    .filter(col("folder_date").between("2026-06-23", "2026-07-22"))
    .drop("folder_date")
)

ticket_1month_df.printSchema()
ticket_1month_df.show(5)

# itsm_base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/itsm_hpd_tms_data_hourly"

# # Read
# ticket_raw_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(f"{itsm_base_path}/*/*/*")
#     .withColumn(
#         "file_path",
#         F.col("_metadata.file_path")
#     )
#     .withColumn(
#         "folder_date",
#         regexp_extract(
#             col("file_path"),
#             r"(\d{4}-\d{2}-\d{2})",
#             1
#         )
#     )
#     .filter(
#         col("folder_date").between(
#             "2026-06-23",
#             "2026-07-22"
#         )
#     )
# )

# # Count distinct files
# num_files = (
#     ticket_raw_df
#     .select("file_path")
#     .distinct()
#     .count()
# )

# print("Number of files read:", num_files)

# # Final dataframe - remove helper columns
# ticket_raw_df = (
#     ticket_raw_df
#     .drop("folder_date", "file_path")
# )

# print("Rows:", ticket_raw_df.count())
# print("Columns:", len(ticket_raw_df.columns))

root
 |-- INCIDENT_NUMBER: string (nullable = true)
 |-- VENDOR_TICKET_NUMBER: string (nullable = true)
 |-- ASSIGNED_GROUP: string (nullable = true)
 |-- IMPACT: integer (nullable = true)
 |-- PRIORITY: integer (nullable = true)
 |-- URGENCY: integer (nullable = true)
 |-- COMPANY: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- DETAILED_DESCRIPTION: string (nullable = true)
 |-- HPD_CI: string (nullable = true)
 |-- NODETYPE: string (nullable = true)
 |-- OP_TIME: timestamp (nullable = true)
 |-- OP_TYPE: string (nullable = true)
 |-- SUBMIT_DATE: timestamp (nullable = true)
 |-- submit_date_UTC: timestamp (nullable = true)
 |-- submit_date_BKK: timestamp (nullable = true)
 |-- Categorization_Tier_1: string (nullable = true)
 |-- Categorization_Tier_2: string (nullable = true)
 |-- Categorization_Tier_3: string (nullable = true)
 |-- RESOLUTION_CATEGORY: string (nullable = true)
 |-- RESOLUTION_CATEGORY_TIER_2: string (nullable = true)
 |-- RESOLUTION_CATEGOR

In [21]:
print("Rows:", ticket_1month_df.count())
print("Columns:", len(ticket_1month_df.columns))

Rows: 22167499
Columns: 31


In [48]:
itsm_closed_base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/itsm-closed-data"

ticket_1month_closed_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(f"{itsm_closed_base_path}/*/*.csv")
    .withColumn("folder_date", regexp_extract(col("_metadata.file_path"), r"(\d{4}-\d{2}-\d{2})", 1))
    .filter(col("folder_date").between("2026-06-23", "2026-07-22"))
    .drop("folder_date")
)

ticket_1month_closed_df.printSchema()
ticket_1month_closed_df.show(5)

root
 |-- INCIDENT_NUMBER: string (nullable = true)
 |-- VENDOR_TICKET_NUMBER: string (nullable = true)
 |-- ASSIGNED_GROUP: string (nullable = true)
 |-- IMPACT: integer (nullable = true)
 |-- PRIORITY: integer (nullable = true)
 |-- URGENCY: integer (nullable = true)
 |-- COMPANY: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- DETAILED_DESCRIPTION: string (nullable = true)
 |-- HPD_CI: string (nullable = true)
 |-- NODETYPE: string (nullable = true)
 |-- OP_TIME: timestamp (nullable = true)
 |-- OP_TYPE: string (nullable = true)
 |-- SUBMIT_DATE: timestamp (nullable = true)
 |-- submit_date_UTC: timestamp (nullable = true)
 |-- submit_date_BKK: timestamp (nullable = true)
 |-- Categorization_Tier_1: string (nullable = true)
 |-- Categorization_Tier_2: string (nullable = true)
 |-- Categorization_Tier_3: string (nullable = true)
 |-- RESOLUTION_CATEGORY: string (nullable = true)
 |-- RESOLUTION_CATEGORY_TIER_2: string (nullable = true)
 |-- RESOLUTION_CATEGOR

In [16]:
ticket_1month_closed_df.count()

31396

In [17]:
ticket_1month_closed_df.groupBy("NODETYPE") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+--------------------------------+-----+
|NODETYPE                        |count|
+--------------------------------+-----+
|IPRAN                           |10906|
|Node B                          |3314 |
|LTE Cell                        |2922 |
|eNode B                         |2903 |
|GGSN - GPRS Gateway Support Node|1833 |
|BTS - Base Tranceiver Station   |1073 |
|IDD                             |1042 |
|IPRAN DN                        |926  |
|UMTS Cell                       |905  |
|IPRAN AGG                       |885  |
|Other                           |547  |
|null                            |544  |
|GSM Cell                        |466  |
|DWDM - Lucent                   |378  |
|NR GNodeB - Huawei              |276  |
|NR GNodeB - Ericsson            |266  |
|PAC (Wind Cooling)              |264  |
|IP Router                       |203  |
|Transmission - Microwave        |177  |
|DNA                             |147  |
+--------------------------------+-----+
only showing top

In [22]:
ticket_1month_df \
    .filter(F.col("NODETYPE") == "NR GNodeB - Ericsson") \
    .groupBy("NODETYPE") \
    .count() \
    .show(truncate=False)

+--------------------+------+
|NODETYPE            |count |
+--------------------+------+
|NR GNodeB - Ericsson|144931|
+--------------------+------+



In [23]:
#Read week by week and union
onefm_base_path = "/persistent/RAN_fault_v1/RAN_fault_v2/dnb_ibo_data_trasfer/onefm"

def read_week(start, end):
    return (
        spark.read
        .option("header", "true")
        .option("sep", "~")
        .csv(f"{onefm_base_path}/*/*/*")
        .withColumn("folder_date", regexp_extract(col("_metadata.file_path"), r"(\d{4}-\d{2}-\d{2})", 1))
        .filter(col("folder_date").between(start, end))
        .drop("folder_date")
    )

week1 = read_week("2026-06-23", "2026-06-29")
week2 = read_week("2026-06-30", "2026-07-06")
week3 = read_week("2026-07-07", "2026-07-13")
week4 = read_week("2026-07-14", "2026-07-22")

alarm_1month_df = week1.union(week2).union(week3).union(week4)


In [24]:
alarm_1month_df.printSchema()

root
 |-- RECTIMESTAMP: string (nullable = true)
 |-- FIRSTOCCURRENCE: string (nullable = true)
 |-- IDENTIFIER: string (nullable = true)
 |-- LASTOCCURRENCE: string (nullable = true)
 |-- SERVICE: string (nullable = true)
 |-- EMS_NAME: string (nullable = true)
 |-- NODESTATUS: string (nullable = true)
 |-- SERVICEAFFECTING: string (nullable = true)
 |-- CLEARTIME: string (nullable = true)
 |-- OBJECT: string (nullable = true)
 |-- NODETYPE: string (nullable = true)
 |-- SEVERITY: string (nullable = true)
 |-- EXTENDEDATTR: string (nullable = true)
 |-- ELEMENTMANAGERIP: string (nullable = true)
 |-- EVENTID: string (nullable = true)
 |-- ADDITIONALSTRING: string (nullable = true)
 |-- ALERTKEY: string (nullable = true)
 |-- SITENAME: string (nullable = true)
 |-- ORIGINALSEVERITY: string (nullable = true)
 |-- SITESTATUS: string (nullable = true)
 |-- SITEID: string (nullable = true)
 |-- CLASS: string (nullable = true)
 |-- OBJECTSTATUS: string (nullable = true)
 |-- CLEAREDBY: stri

In [62]:
alarm_1month_df

DataFrame[RECTIMESTAMP: string, FIRSTOCCURRENCE: string, IDENTIFIER: string, LASTOCCURRENCE: string, SERVICE: string, EMS_NAME: string, NODESTATUS: string, SERVICEAFFECTING: string, CLEARTIME: string, OBJECT: string, NODETYPE: string, SEVERITY: string, EXTENDEDATTR: string, ELEMENTMANAGERIP: string, EVENTID: string, ADDITIONALSTRING: string, ALERTKEY: string, SITENAME: string, ORIGINALSEVERITY: string, SITESTATUS: string, SITEID: string, CLASS: string, OBJECTSTATUS: string, CLEAREDBY: string, X733SPECIFICPROB: string, MANAGER: string, NODEALIAS: string, SUMMARY: string, NODE: string, TT_FLAG: string, SERVERSERIAL: string, TMTRANSSTATE: string, TTCREATIONTIME: string, TMTRANSBEGINTIME: string, TT_ID: string, RECEIVEDATPROBE: string, physicalCard: string, OP_TYPE: string, OP_TIME_DWH: string, CIRCLE: string, REGION: string, LOCATION: string, BACKHAULTYPE: string, CONTROLNE: string]

In [25]:
ticket_1month_closed_df.show(5)

+---------------+--------------------+--------------------+------+--------+-------+----------+--------------------+--------------------+-------+--------------------+--------------------+-------+-------------------+-------------------+-------------------+---------------------+---------------------+---------------------+--------------------+--------------------------+--------------------------+----------+------+----------------------+----------------------+-------------------+-------------------+---------------+---+--------+
|INCIDENT_NUMBER|VENDOR_TICKET_NUMBER|      ASSIGNED_GROUP|IMPACT|PRIORITY|URGENCY|   COMPANY|         DESCRIPTION|DETAILED_DESCRIPTION| HPD_CI|            NODETYPE|             OP_TIME|OP_TYPE|        SUBMIT_DATE|    submit_date_UTC|    submit_date_BKK|Categorization_Tier_1|Categorization_Tier_2|Categorization_Tier_3| RESOLUTION_CATEGORY|RESOLUTION_CATEGORY_TIER_2|RESOLUTION_CATEGORY_TIER_3|INC_STATUS|STATUS|LAST_RESOLVED_DATE_UTC|LAST_RESOLVED_DATE_BKK|    CLOSED_D

In [27]:
print(alarm_1month_df.count())

240221339


In [49]:
#to keep only 5g data
ticket_5G_df = ticket_1month_df.filter(
    F.col("NODETYPE") == "NR GNodeB - Ericsson"
)

print(f"Rows after filtering: {ticket_5G_df.count():,}")

ticket_5G_df.show(5, truncate=False)

Rows after filtering: 144,931
+---------------+---------------------+---------------------+------+--------+-------+----------+------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------------------+--------------------------+-------+-------------------+-------------------+-------------------+---------------------+---------------------+---------------------+-------------------------------------------------

In [50]:
#to keep only 5g data

ticket_5G_closed_df = ticket_1month_closed_df.filter(
    F.col("NODETYPE") == "NR GNodeB - Ericsson"
)

print(f"Rows after filtering: {ticket_5G_closed_df.count():,}")

ticket_5G_closed_df.show(5, truncate=False)

Rows after filtering: 266
+---------------+-----------------------------+------------------------+------+--------+-------+----------+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------------------+--------------------------+-------+-------------------+-------------------+-------------------+---------------------+----------------------------+---------------------+------------------------

In [51]:
# Keep latest record for each HPD_CI + INCIDENT_NUMBER
# based on EVENT_START_TIME
from pyspark.sql.window import Window
from pyspark.sql import functions as F
window_ticket = (
    Window
    .partitionBy("HPD_CI", "INCIDENT_NUMBER")
    .orderBy(F.col("EVENT_START_TIME").desc())
)

ticket_5G_df = (
    ticket_5G_df
    .withColumn(
        "rn",
        F.row_number().over(window_ticket)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [52]:
window_closed = (
    Window
    .partitionBy("HPD_CI", "INCIDENT_NUMBER")
    .orderBy(F.col("LAST_RESOLVED_DATE_UTC").desc())
)

ticket_5G_closed_df = (
    ticket_5G_closed_df
    .withColumn("rn", F.row_number().over(window_closed))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [53]:
closed_cols = [
    "INCIDENT_NUMBER",
    "LAST_RESOLVED_DATE_UTC",
    "LAST_RESOLVED_DATE_BKK",
    "CLOSED_DATE_UTC",
    "CLOSED_DATE_BKK"
]

merged_ticket_df = (
    ticket_5G_df.alias("t")
    .join(
        ticket_5G_closed_df.select(*closed_cols).alias("c"),
        on="INCIDENT_NUMBER",
        how="left"
    )
)

In [22]:
print(merged_ticket_df.columns)

['INCIDENT_NUMBER', 'VENDOR_TICKET_NUMBER', 'ASSIGNED_GROUP', 'IMPACT', 'PRIORITY', 'URGENCY', 'COMPANY', 'DESCRIPTION', 'DETAILED_DESCRIPTION', 'HPD_CI', 'NODETYPE', 'OP_TIME', 'OP_TYPE', 'SUBMIT_DATE', 'submit_date_UTC', 'submit_date_BKK', 'Categorization_Tier_1', 'Categorization_Tier_2', 'Categorization_Tier_3', 'RESOLUTION_CATEGORY', 'RESOLUTION_CATEGORY_TIER_2', 'RESOLUTION_CATEGORY_TIER_3', 'INC_STATUS', 'STATUS', 'REPORTED_SOURCE', 'EVENT_END_TIME', 'EVENT_START_TIME', 'PENDING_TIME', 'PENDING_DUE_TIME', 'SEVERITY', 'LAST_RESOLVED_DATE_UTC', 'LAST_RESOLVED_DATE_BKK', 'CLOSED_DATE_UTC', 'CLOSED_DATE_BKK']


In [ ]:
# # ==========================================================
# # Find degraded cells (minimum availability < 100)
# # ==========================================================
# degraded_cells = (
#     combined_df
#     .groupBy("CELLID")
#     .agg(F.min("NR_Cell_Availability").alias("min_availability")) 
#     .filter(F.col("min_availability") < 100)
#     .select("CELLID")
# )

# # Show first few rows
# degraded_cells.show(5, truncate=False)

# # Number of degraded cells
# print("Number of degraded cells:", degraded_cells.count())

# # ==========================================================
# # Keep only records belonging to degraded cells
# # ==========================================================
# filtered_df = (
#     combined_df
#     .join(degraded_cells, on="CELLID", how="inner")
# )

In [54]:
# ==========================================================
# Map NODETYPE -> Technology
# ==========================================================
filtered_ticket_df = (
    merged_ticket_df
    .withColumn(
        "Technology",
        F.when(F.col("NODETYPE").isin("eNode B", "LTE Cell"), "4G")
         .when(F.col("NODETYPE") == "NR GNodeB - Ericsson", "5G")
         .otherwise(None)
    )
)

# ==========================================================
# Apply filters
# ==========================================================
filtered_ticket_df = (
    filtered_ticket_df
    .filter(
        # Keep only 4G and 5G Ericsson records
        F.col("Technology").isin("4G", "5G")

        # Assigned Group contains BBT or WW
        & F.col("ASSIGNED_GROUP").rlike("(?i)BBT|WW")

        # Severity contains SA1 or SA2 or SA3
        & F.col("SEVERITY").rlike("(?i)SA1|SA2|SA3")

        # Categorization Tier 1
        & (F.col("Categorization_Tier_1") == "NOC-NW-RAN")

        # Categorization Tier 2
        & F.col("Categorization_Tier_2").isin(
            "RAN-ERICSSON SITE DOWN",
            "RAN-ERICSSON SITE UP/DOWN",
            "RAN-ERICSSON ROUTE SITE DOWN"
        )
    )
)

# ==========================================================
# Technology counts (equivalent to value_counts())
# ==========================================================
filtered_ticket_df.groupBy("Technology").count().show()

+----------+-----+
|Technology|count|
+----------+-----+
|        5G|  143|
+----------+-----+



In [43]:
filtered_ticket_df.count()

143

In [138]:
# ticket_pd = filtered_ticket_df.toPandas()


In [139]:
# output_file_ticket = "ticket_df_filtered.csv"

# ticket_pd.to_csv(
#     output_file_ticket,
#     index=False
# )

# print("Saved to:", os.path.abspath(output_file_ticket))
# print(f"Rows saved: {len(ticket_pd):,}")
# print(f"Columns saved: {len(ticket_pd.columns)}")

In [55]:
site_list = (
    filtered_ticket_df
    .select("HPD_CI")
    .where(F.col("HPD_CI").isNotNull())
    .distinct()
    .rdd.flatMap(lambda x: x)
    .collect()
)

print(site_list)

['KCN0330', 'CMI7624', 'AYT0369', 'SMS8539', 'SMK7143', 'LPN1909', 'SRB7286', 'LBR0176', 'RCB7263', 'SMK8618', 'RCB6744', 'CMI2019', 'NKT0260', 'RCB8620', 'NKT0266', 'KCNC008', 'NKT0295', 'RCB8568', 'SRB0068', 'RCB1003', 'CMI0194', 'AYT7193', 'AYT6773', 'CMI6865', 'RCB8514', 'PYO1699', 'NKW0330', 'AYT7432', 'SMS0048', 'NKW0139', 'NKT7244', 'SRBEV83', 'PCB0168', 'RCB0222', 'AYT0143', 'AYT0425', 'SMK0254', 'TAK6704', 'SPB0126', 'CNT8521', 'RCB0118', 'CRI6320', 'KCN6723', 'RCB7173', 'NKW8534', 'CNT0039', 'LPN8633', 'UTR6727', 'NKT0089', 'SMK7671', 'ATG0096', 'SMK0024', 'KCN0293', 'AYT0550', 'SPB8540', 'KCN0016', 'RCB0317', 'SPB0019', 'SPB0270', 'CMI1672', 'NKW1925', 'RCB0332', 'TAK6710', 'SPB0164', 'SRB0047', 'SMK0295', 'SMK0474', 'LPN7167', 'KCN0354', 'LPG6739', 'NKW0129', 'NKT0347', 'KCN0143', 'SMK7217', 'LBR0064', 'CRI7175', 'NKW1603', 'AYT0405', 'RCB6738', 'SMS0045', 'PCT3036', 'CMI6294', 'AYT0347', 'PSN7252', 'RCBC011', 'RCB0353', 'AYT8652', 'SMK0255', 'SPB0160', 'KCNC004', 'TAK6717'

In [35]:
alarm_1month_df.groupBy("NODETYPE") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+------------------------+--------+
|NODETYPE                |count   |
+------------------------+--------+
|OLT                     |79828635|
|IPRAN                   |25027016|
|eNode B                 |21762569|
|null                    |18178406|
|Node B                  |12965495|
|DWDM                    |6655663 |
|LTE Cell                |5914367 |
|IPRAN AGG               |5619715 |
|Transmission - Microwave|4286474 |
|NR GNodeB - Huawei      |4233704 |
|Firewall                |4071131 |
|TICORP                  |3157978 |
|Other                   |3135696 |
|IPRAN DN                |2745958 |
|UMTS Cell               |2601094 |
|Switch                  |2497364 |
|MNS                     |2497240 |
|NR GNodeB - Ericsson    |2393176 |
|EDFA-OLT                |1918152 |
|MPLS                    |1819106 |
+------------------------+--------+
only showing top 20 rows



In [36]:
# Keep only NR GNodeB - Ericsson rows
alarm_1month_df = alarm_1month_df.filter(
    F.col("NODETYPE") == "NR GNodeB - Ericsson"
)

# Verify the result
print(f"Rows after filtering: {alarm_1month_df.count():,}")

Rows after filtering: 2,393,176


In [73]:
null_counts = alarm_1month_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in alarm_1month_df.columns
])

null_counts.show(truncate=False)

+------------+---------------+----------+--------------+-------+--------+----------+----------------+---------+-------+--------+--------+------------+----------------+-------+----------------+--------+--------+----------------+----------+------+------+------------+---------+----------------+-------+---------+-------+------+-------+------------+------------+--------------+----------------+-------+---------------+------------+-------+-----------+------+------+--------+------------+---------+
|RECTIMESTAMP|FIRSTOCCURRENCE|IDENTIFIER|LASTOCCURRENCE|SERVICE|EMS_NAME|NODESTATUS|SERVICEAFFECTING|CLEARTIME|OBJECT |NODETYPE|SEVERITY|EXTENDEDATTR|ELEMENTMANAGERIP|EVENTID|ADDITIONALSTRING|ALERTKEY|SITENAME|ORIGINALSEVERITY|SITESTATUS|SITEID|CLASS |OBJECTSTATUS|CLEAREDBY|X733SPECIFICPROB|MANAGER|NODEALIAS|SUMMARY|NODE  |TT_FLAG|SERVERSERIAL|TMTRANSSTATE|TTCREATIONTIME|TMTRANSBEGINTIME|TT_ID  |RECEIVEDATPROBE|physicalCard|OP_TYPE|OP_TIME_DWH|CIRCLE|REGION|LOCATION|BACKHAULTYPE|CONTROLNE|
+---------

In [66]:
# alarm_1month_df
alarm_1month_df.select("RECTIMESTAMP").distinct().show(truncate=False)

+-------------------+
|RECTIMESTAMP       |
+-------------------+
|06/23/2026 09:35:03|
|06/23/2026 09:30:03|
|06/23/2026 09:25:02|
|06/25/2026 09:00:03|
|06/23/2026 09:15:02|
|06/28/2026 05:45:02|
|06/25/2026 08:40:03|
|06/24/2026 12:40:02|
|06/29/2026 00:10:02|
|06/27/2026 05:55:03|
|06/23/2026 09:55:02|
|06/23/2026 05:45:03|
|06/28/2026 11:45:02|
|06/25/2026 14:25:02|
|06/27/2026 12:40:03|
|06/23/2026 00:10:02|
|06/28/2026 09:10:03|
|06/28/2026 09:20:02|
|06/25/2026 09:10:02|
|06/23/2026 10:10:02|
+-------------------+
only showing top 20 rows



In [67]:
# alarm_1month_df
alarm_1month_df.select("RECEIVEDATPROBE").distinct().show(truncate=False)

+-----------------------+
|RECEIVEDATPROBE        |
+-----------------------+
|2026-06-23 09:24:45 UTC|
|2026-06-23 09:27:11 UTC|
|2026-06-23 09:28:42 UTC|
|2026-06-23 00:19:14 UTC|
|2026-06-23 08:29:02 UTC|
|2026-06-23 09:18:21 UTC|
|2026-06-22 23:33:12 UTC|
|2026-06-23 09:19:47 UTC|
|2026-06-21 23:24:24 UTC|
|2026-06-23 09:19:48 UTC|
|2026-06-23 09:18:05 UTC|
|2026-06-23 09:27:00 UTC|
|2026-06-23 09:22:09 UTC|
|2026-06-22 22:54:21 UTC|
|2026-06-23 00:13:08 UTC|
|2026-06-23 09:27:49 UTC|
|2026-06-23 09:23:49 UTC|
|2026-06-23 08:53:27 UTC|
|2026-06-23 09:19:49 UTC|
|2026-06-23 09:29:18 UTC|
+-----------------------+
only showing top 20 rows



In [68]:
from pyspark.sql import functions as F

# Convert RECTIMESTAMP to timestamp
alarm_1month_df = alarm_1month_df.withColumn(
    "RECTIMESTAMP",
    F.to_timestamp("RECTIMESTAMP", "MM/dd/yyyy HH:mm:ss")
)

# Convert RECEIVEDATPROBE to timestamp
alarm_1month_df = alarm_1month_df.withColumn(
    "RECEIVEDATPROBE",
    F.to_timestamp("RECEIVEDATPROBE", "yyyy-MM-dd HH:mm:ss 'UTC'")
)

# Fill null RECEIVEDATPROBE values using RECTIMESTAMP
alarm_1month_df = alarm_1month_df.withColumn(
    "RECEIVEDATPROBE",
    F.coalesce(F.col("RECEIVEDATPROBE"), F.col("RECTIMESTAMP"))
)

In [69]:
from pyspark.sql import functions as F

alarm_1month_df.select(
    F.count(
        F.when(F.col("RECEIVEDATPROBE").isNull(), 1)
    ).alias("null_count")
).show()

+----------+
|null_count|
+----------+
|         0|
+----------+



In [70]:
alarm_df = alarm_1month_df.filter(
    F.col("SITEID").isNotNull() &
    F.col("NODE").isNotNull()
)

print(f"Number of rows after dropping NULL SITEID/NODE: {alarm_df.count()}")

Number of rows after dropping NULL SITEID/NODE: 2044322


In [71]:
site_ids=site_list

# Filter the DataFrame
alarm_df_filtered = alarm_df.filter(
    F.col("SITEID").isin(site_ids)
)

print(f"Rows after filtering: {alarm_df_filtered.count():,}")

Rows after filtering: 66,560


In [30]:
kpi_1month_5G_df_copy = kpi_1month_5G_df

In [31]:
kpi_1month_5G_df_copy = (
    kpi_1month_5G_df_copy
    .filter(
        F.substring(F.col("SITE"), 1, 7).isin(site_ids)
    )
)

print(f"Rows after filtering: {kpi_1month_5G_df_copy.count():,}")

Rows after filtering: 1,201,266


In [ ]:
# Filter the DataFrame
# alarm_df_filtered = alarm_df.filter(
#     F.col("SITEID").isin(site_ids)
# )

# print(f"Rows after filtering: {alarm_df_filtered.count():,}")

In [32]:
(
    alarm_df_filtered
    .groupBy("EMS_Name")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-----------------------------------+-----+
|EMS_Name                           |count|
+-----------------------------------+-----+
|True_Ericsson_ENM_RAN_6A           |20617|
|True_Ericsson_ENM2_WEST            |15878|
|True_Ericsson_ENM_RAN_7A           |13087|
|True_Ericsson_ENM5_WEST_UPPER_NORTH|8795 |
|True_Ericsson_ENM_UTRAN_CENTRAL_4A |4602 |
|True_Ericsson_ENM_UTRAN_SOUTH_3A   |2907 |
|True_Ericsson_ENM_RAN_North_1A     |666  |
|UTRAN-CENTRAL-ENM4A                |8    |
+-----------------------------------+-----+



In [88]:
# alarm_pd = alarm_df_filtered.toPandas()

In [89]:
# output_file = "alarm_df_filtered.csv"

# alarm_pd.to_csv(
#     output_file,
#     index=False
# )

# print("Saved to:", os.path.abspath(output_file))
# print(f"Rows saved: {len(alarm_pd):,}")
# print(f"Columns saved: {len(alarm_pd.columns)}")

Saved to: /persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month/alarm_df_filtered.csv
Rows saved: 66,492
Columns saved: 44


In [146]:
# filtered_ticket_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("/persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month/ticket_df_filtered.csv")
# )

# filtered_ticket_df.show(5)
# filtered_ticket_df.printSchema()

In [147]:
# # Number of rows
# num_rows = filtered_ticket_df.count()

# # Number of columns
# num_cols = len(filtered_ticket_df.columns)

# print(f"Shape: ({num_rows:,}, {num_cols})")

In [152]:
from pyspark.sql import functions as F

filtered_ticket_df.groupBy("HPD_CI") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

+-------+-----+
|HPD_CI |count|
+-------+-----+
|CMI7624|7    |
|CRI6320|3    |
|PCB0168|2    |
|RCB1003|2    |
|SPB8540|2    |
|TAK6704|2    |
|PCB7676|2    |
|AYT7193|2    |
|AYT0550|2    |
|TAK6710|2    |
|KCN0143|2    |
|SMS0045|2    |
|CMI8863|2    |
|SMK7671|2    |
|UTR6727|2    |
|SMK0024|2    |
|NKW0330|2    |
|KCN0330|1    |
|AYT0369|1    |
|SMS8539|1    |
+-------+-----+
only showing top 20 rows



In [148]:
# alarm_df_filtered = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv("/persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month/alarm_df_filtered.csv")
# )

# alarm_pd.show(5)
# alarm_pd.printSchema()

In [154]:
# Number of rows
num_rows = alarm_df_filtered.count()

# Number of columns
num_cols = len(alarm_df_filtered.columns)

print(f"Shape: ({num_rows:,}, {num_cols})")

Shape: (66,492, 45)


In [72]:
summary_clean = F.trim(
    F.regexp_replace(F.col("SUMMARY"), r"\s*:.*$", "")
)

# Normalize function
def normalize(col):
    return F.regexp_replace(
        F.lower(F.trim(col)),
        r"[^a-z0-9]+",
        ""
    )

x733_norm = normalize(F.col("X733SPECIFICPROB"))
summary_norm = normalize(summary_clean)

alarm_df_filtered = alarm_df_filtered.withColumn(
    "ALARM_DESCRIPTION",
    F.when(
        x733_norm == summary_norm,
        F.col("X733SPECIFICPROB")
    ).otherwise(
        F.concat_ws(
            " | ",
            F.col("X733SPECIFICPROB"),
            summary_clean
        )
    )
)

In [80]:
# alarm_df_filtered = alarm_df_filtered.filter(
#     ~(
#         F.lower(F.col("ALARM_DESCRIPTION")).contains("alarm database upload in progress")
#         &
#         F.col("CLEAREDBY").isNull()
#     )
# )

In [81]:
# alarm_df_filtered.filter(
#     F.col("ALARM_DESCRIPTION").rlike("(?i)ALARM DATABASE UPLOAD IN PROGRESS")
# ).select("CLEAREDBY").distinct().show(truncate=False)

+---------+
|CLEAREDBY|
+---------+
+---------+



In [73]:
from pyspark.sql import functions as F
from itertools import chain

# Dictionary
nodetype_tech_mapping = {
    "eNode B": "4G",
    "LTE Cell": "4G",
    "NR GNodeB - Ericsson": "5G"
}

# Create Spark map expression
mapping_expr = F.create_map(
    *[F.lit(x) for x in chain(*nodetype_tech_mapping.items())]
)

# Map NODETYPE -> Technology
alarm_df_filtered = (
    alarm_df_filtered
    .withColumn("Technology", mapping_expr[F.col("NODETYPE")])
    .filter(F.col("Technology").isNotNull())
)

# Technology counts
alarm_df_filtered.groupBy("Technology").count().show()

+----------+-----+
|Technology|count|
+----------+-----+
|        5G|66560|
+----------+-----+



In [166]:
alarm_df_filtered.select("LASTOCCURRENCE").distinct().show(truncate=False)

+-------------------+
|LASTOCCURRENCE     |
+-------------------+
|06/25/2026 08:41:36|
|06/25/2026 08:45:29|
|06/25/2026 08:45:57|
|06/25/2026 08:45:22|
|06/25/2026 08:43:34|
|06/25/2026 08:45:24|
|06/25/2026 07:58:57|
|06/25/2026 08:22:55|
|06/26/2026 14:15:09|
|06/25/2026 07:44:55|
|06/27/2026 05:50:29|
|06/25/2026 08:22:44|
|06/27/2026 05:53:35|
|06/25/2026 12:34:19|
|06/28/2026 09:07:44|
|06/25/2026 12:30:15|
|06/28/2026 09:09:07|
|06/28/2026 09:04:48|
|06/28/2026 09:05:54|
|06/28/2026 09:07:10|
+-------------------+
only showing top 20 rows



In [74]:
group_cols = [
    "SITEID",
    "Technology",
    "RECTIMESTAMP",
    "RECEIVEDATPROBE",
    "CLEARTIME",
    "X733SPECIFICPROB",
]

# Ensure LASTOCCURRENCE is a timestamp (if it's not already)

alarm_df_filtered = alarm_df_filtered.withColumn(
    "LASTOCCURRENCE",
    F.to_timestamp(
        F.col("LASTOCCURRENCE"),
        "MM/dd/yyyy HH:mm:ss"
    )
)

# Window: latest LASTOCCURRENCE within each group
window_spec = (
    Window
    .partitionBy(*group_cols)
    .orderBy(F.col("LASTOCCURRENCE").desc_nulls_last())
)

# Keep only the latest record in each group
alarm_df_filtered = (
    alarm_df_filtered
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Shape:", (alarm_df_filtered.count(), len(alarm_df_filtered.columns)))

display(alarm_df_filtered)

Shape: (46826, 46)


DataFrame[RECTIMESTAMP: timestamp, FIRSTOCCURRENCE: string, IDENTIFIER: string, LASTOCCURRENCE: timestamp, SERVICE: string, EMS_NAME: string, NODESTATUS: string, SERVICEAFFECTING: string, CLEARTIME: string, OBJECT: string, NODETYPE: string, SEVERITY: string, EXTENDEDATTR: string, ELEMENTMANAGERIP: string, EVENTID: string, ADDITIONALSTRING: string, ALERTKEY: string, SITENAME: string, ORIGINALSEVERITY: string, SITESTATUS: string, SITEID: string, CLASS: string, OBJECTSTATUS: string, CLEAREDBY: string, X733SPECIFICPROB: string, MANAGER: string, NODEALIAS: string, SUMMARY: string, NODE: string, TT_FLAG: string, SERVERSERIAL: string, TMTRANSSTATE: string, TTCREATIONTIME: string, TMTRANSBEGINTIME: string, TT_ID: string, RECEIVEDATPROBE: timestamp, physicalCard: string, OP_TYPE: string, OP_TIME_DWH: string, CIRCLE: string, REGION: string, LOCATION: string, BACKHAULTYPE: string, CONTROLNE: string, ALARM_DESCRIPTION: string, Technology: string]

In [36]:
filtered_df=kpi_1month_5G_df_copy

In [37]:
filtered_df = filtered_df.select(
    "COLLECTTIME",
    "CELLID",
    "SITE",
    "NR_Cell_Availability"
)

In [38]:
from pyspark.sql import functions as F

# ============================================================================
# 1. Drop columns with >50% missing values
# ============================================================================

# Total number of rows
total_rows = filtered_df.count()

# Count nulls in each column
null_counts = filtered_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in filtered_df.columns
]).first().asDict()

# Keep columns with <= 50% null values
cols_to_keep = [
    c for c, null_count in null_counts.items()
    if null_count <= 0.5 * total_rows
]

filtered_df = filtered_df.select(*cols_to_keep)

print("Columns after dropping:", len(filtered_df.columns))

# ============================================================================
# 2. Remove duplicate rows
# ============================================================================

filtered_df = filtered_df.dropDuplicates()

print("Rows after removing duplicates:", filtered_df.count())

Columns after dropping: 4


Rows after removing duplicates: 1201266


In [75]:
#Get unique SITE prefixes (first 7 characters)
site_df = (
    filtered_df
    .select(F.substring("SITE", 1, 7).alias("SITEID"))
    .where(F.col("SITEID").isNotNull())
    .distinct()
)

# Keep only alarms whose SITEID exists in the KPI data
alarm_df_filtered = (
    alarm_df_filtered
    .join(site_df, on="SITEID", how="inner")
)

In [76]:
from pyspark.sql import functions as F

timestamp_format = "MM/dd/yyyy HH:mm:ss"

alarm_df_filtered = (
    alarm_df_filtered
    .withColumn(
        "FIRSTOCCURRENCE",
        F.to_timestamp("FIRSTOCCURRENCE", timestamp_format)
    )
)

filtered_ticket_df = (
    filtered_ticket_df
    .withColumn(
        "EVENT_START_TIME",
        F.to_timestamp("EVENT_START_TIME", timestamp_format)
    )
    .withColumn(
        "EVENT_END_TIME",
        F.to_timestamp("EVENT_END_TIME", timestamp_format)
    )
)

print("✅ Timestamp conversion completed.")

✅ Timestamp conversion completed.


In [41]:
filtered_df = filtered_df.withColumn(
    "SITEID",
    F.substring("SITE", 1, 7)
)

In [78]:
from pyspark.sql import functions as F

# ============================================================
# STEP 2: SITE ID ALIGNMENT ACROSS ALL THREE
# ============================================================

# Show sample SITEIDs
print("\nKPI SITEID sample:")
filtered_df.select("SITEID").distinct().show(5, truncate=False)

print("Alarm SITEID sample:")
alarm_df_filtered.select("SITEID").distinct().show(5, truncate=False)

print("Ticket HPD_CI sample:")
filtered_ticket_df.select("HPD_CI").distinct().show(5, truncate=False)

# ============================================================
# STEP 3: FILTER ALARM & TICKET TO ONLY KPI SITES
# ============================================================

# KPI sites (keep as a Spark DataFrame)
kpi_sites_df = (
    filtered_df
    .select("SITEID")
    .where(F.col("SITEID").isNotNull())
    .distinct()
)

# Filter alarm data
alarm_filtered = (
    alarm_df_filtered
    .join(kpi_sites_df, on="SITEID", how="inner")
)

# Filter ticket data
ticket_filtered = (
    filtered_ticket_df
    .join(
        kpi_sites_df.withColumnRenamed("SITEID", "HPD_CI"),
        on="HPD_CI",
        how="inner"
    )
)

# ============================================================
# Summary
# ============================================================

print("\nKPI unique sites:", kpi_sites_df.count())

print(
    "Alarm rows after filter:",
    alarm_filtered.count(),
    ", unique sites:",
    alarm_filtered.select("SITEID").distinct().count()
)

print(
    "Ticket rows after filter:",
    ticket_filtered.count(),
    ", unique sites:",
    ticket_filtered.select("HPD_CI").distinct().count()
)


KPI SITEID sample:


+-------+
|SITEID |
+-------+
|NKT0310|
|UTR6727|
|NKT0089|
|RCB0353|
|AYT8652|
+-------+
only showing top 5 rows

Alarm SITEID sample:


+-------+
|SITEID |
+-------+
|NKT0310|
|UTR6727|
|NKT0089|
|RCB0353|
|AYT8652|
+-------+
only showing top 5 rows

Ticket HPD_CI sample:


+-------+
|HPD_CI |
+-------+
|KCN0330|
|CMI7624|
|AYT0369|
|SMS8539|
|SMK7143|
+-------+
only showing top 5 rows




KPI unique sites: 119


Alarm rows after filter: 46826 , unique sites: 118


Ticket rows after filter: 142 , unique sites: 119


In [43]:
# # ============================================================
# # Output Directory
# # ============================================================

# OUTPUT_DIR = "/persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month"

# # ============================================================
# # Save filtered_ticket_df
# # ============================================================

# (
#     ticket_filtered
#     .coalesce(1)              # Write a single CSV file
#     .write
#     .mode("overwrite")
#     .option("header", "true")
#     .csv(f"{OUTPUT_DIR}/ticket_df_filtered.csv")
# )

# # ============================================================
# # Save alarm_filtered
# # ============================================================

# (
#     alarm_filtered
#     .coalesce(1)
#     .write
#     .mode("overwrite")
#     .option("header", "true")
#     .csv(f"{OUTPUT_DIR}/alarm_df_filtered.csv")
# )

# # ============================================================
# # Save filtered_df
# # ============================================================

# (
#     filtered_df
#     .coalesce(1)
#     .write
#     .mode("overwrite")
#     .option("header", "true")
#     .csv(f"{OUTPUT_DIR}/filtered_df.csv")
# )

# print("All DataFrames saved successfully.")

In [44]:
# ============================================================
# STEP 4: REMOVE ONGOING TICKETS (no EVENT_START_TIME)
# ============================================================

ticket_valid = (
    ticket_filtered
    .filter(F.col("EVENT_START_TIME").isNotNull())
)

print("\nTickets with valid EVENT_START_TIME:", ticket_valid.count())

print(
    "Tickets dropped (no start time):",
    ticket_filtered.count() - ticket_valid.count()
)


Tickets with valid EVENT_START_TIME: 142


Tickets dropped (no start time): 0


In [65]:
print(alarm_filtered.columns)

['SITEID', 'RECTIMESTAMP', 'FIRSTOCCURRENCE', 'IDENTIFIER', 'LASTOCCURRENCE', 'SERVICE', 'EMS_NAME', 'NODESTATUS', 'SERVICEAFFECTING', 'CLEARTIME', 'OBJECT', 'NODETYPE', 'SEVERITY', 'EXTENDEDATTR', 'ELEMENTMANAGERIP', 'EVENTID', 'ADDITIONALSTRING', 'ALERTKEY', 'SITENAME', 'ORIGINALSEVERITY', 'SITESTATUS', 'CLASS', 'OBJECTSTATUS', 'CLEAREDBY', 'X733SPECIFICPROB', 'MANAGER', 'NODEALIAS', 'SUMMARY', 'NODE', 'TT_FLAG', 'SERVERSERIAL', 'TMTRANSSTATE', 'TTCREATIONTIME', 'TMTRANSBEGINTIME', 'TT_ID', 'RECEIVEDATPROBE', 'physicalCard', 'OP_TYPE', 'OP_TIME_DWH', 'CIRCLE', 'REGION', 'LOCATION', 'BACKHAULTYPE', 'CONTROLNE', 'ALARM_DESCRIPTION', 'Technology']


In [79]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def clean_alarm_description(text):
    """
    Cleans telecom alarm descriptions by removing dynamic metadata while
    preserving the semantic alarm information.
    """

    if text is None:
        return None

    text = str(text).lower()

    # ---------------------------------------------------------
    # Remove everything inside double quotes (JSON/XML fragments)
    # ---------------------------------------------------------
    text = re.sub(r'"[^"]*"', ' ', text)

    # ---------------------------------------------------------
    # Remove IPv4 addresses
    # ---------------------------------------------------------
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', ' ', text)

    # ---------------------------------------------------------
    # Remove filesystem/object paths
    # ---------------------------------------------------------
    text = re.sub(r'/[^;| ]*', ' ', text)

    # ---------------------------------------------------------
    # Remove ALL key=value pairs
    # ---------------------------------------------------------
    text = re.sub(r'\b[a-zA-Z_]+\s*=\s*[^;|]*', ' ', text)

    # ---------------------------------------------------------
    # Remove standalone numbers
    # ---------------------------------------------------------
    text = re.sub(r'\b\d+\b', ' ', text)

    # ---------------------------------------------------------
    # Remove mixed alphanumeric IDs
    # ---------------------------------------------------------
    text = re.sub(r'\b[a-zA-Z]*\d+[a-zA-Z0-9_-]*\b', ' ', text)

    # ---------------------------------------------------------
    # Remove brackets
    # ---------------------------------------------------------
    text = re.sub(r'[\[\]\(\)\{\}]', ' ', text)

    # ---------------------------------------------------------
    # Remove common telecom metadata words
    # ---------------------------------------------------------
    metadata_words = [
        "srcipaddr",
        "dstipaddr",
        "suppldstnipaddr",
        "supplalarminfo",
        "serial_no",
        "unitname",
        "additionalfaultid",
        "path",
        "timeout connecting to",
        "timeout connecting",
        "timeout",
        "plmn",
        "lnbts_parent",
        "lnbts",
        "lncel"
    ]

    for word in metadata_words:
        text = text.replace(word, " ")

    # ---------------------------------------------------------
    # Normalize separators
    # ---------------------------------------------------------
    text = re.sub(r'\|+', '|', text)
    text = re.sub(r';+', ';', text)

    text = re.sub(r'\s*\|\s*', ' | ', text)
    text = re.sub(r'\s*;\s*', '; ', text)

    # ---------------------------------------------------------
    # Collapse whitespace
    # ---------------------------------------------------------
    text = re.sub(r'\s+', ' ', text).strip()

    # ---------------------------------------------------------
    # Remove duplicate alarm names
    # ---------------------------------------------------------
    if "|" in text:

        left, right = text.split("|", 1)

        left = left.strip()
        right = right.strip()

        if right.startswith(left):
            right = right[len(left):].strip(" ;")

        if right:
            text = left + "; " + right
        else:
            text = left

    # ---------------------------------------------------------
    # Remove repeated separators
    # ---------------------------------------------------------
    text = re.sub(r'(;\s*){2,}', '; ', text)
    text = re.sub(r'(\|\s*){2,}', '| ', text)
    text = re.sub(r';\s*;', '; ', text)
    text = re.sub(r'\|\s*\|', '| ', text)
    text = re.sub(r';\s*$', '', text)
    text = re.sub(r'\|\s*$', '', text)

    # ---------------------------------------------------------
    # Final cleanup
    # ---------------------------------------------------------
    text = re.sub(r'\s+', ' ', text).strip(" ;|")

    return text


# Register UDF
clean_alarm_description_udf = F.udf(clean_alarm_description, StringType())

In [80]:
from pyspark.sql import functions as F

alarm_filtered = alarm_filtered.withColumn(
    "ALARM_DESCRIPTION",
    clean_alarm_description_udf(F.col("ALARM_DESCRIPTION"))
)

In [135]:
# from pyspark.sql import functions as F

# time_cols = {
#     "alarm_filtered": {
#         "df": alarm_filtered,
#         "cols": ["LASTOCCURRENCE", "RECTIMESTAMP", "RECEIVEDATPROBE"]
#     },
#     "filtered_ticket_df": {
#         "df": filtered_ticket_df,
#         "cols": ["EVENT_START_TIME"]
#     }
# }

# for df_name, info in time_cols.items():
#     df = info["df"]
#     for col in info["cols"]:
#         total      = df.count()
#         null_count = df.filter(F.col(col).isNull()).count()
#         not_null   = total - null_count

#         print(f"\n{'='*60}")
#         print(f"  [{df_name}]  column: {col}")
#         print(f"  total={total} | not_null={not_null} | null={null_count}")
#         print(f"{'='*60}")

#         df.select(
#             col,
#             F.col(col).isNull().alias("is_null")
#         ).distinct().orderBy(col).show(100, truncate=False)


In [81]:
from pyspark.sql import functions as F

# ============================================================
# Convert Alarm Timestamp Columns
# ============================================================

alarm_filtered = (
    alarm_filtered
    .withColumn(
        "LASTOCCURRENCE",
        F.to_timestamp(F.col("LASTOCCURRENCE"), "MM/dd/yyyy HH:mm:ss")
    )
    .withColumn(
        "RECTIMESTAMP",
        F.to_timestamp(F.col("RECTIMESTAMP"), "MM/dd/yyyy HH:mm:ss")
    )
    .withColumn(
        "RECEIVEDATPROBE",
        F.to_timestamp(F.col("RECEIVEDATPROBE"), "MM/dd/yyyy HH:mm:ss")
    )
)

# ============================================================
# Convert Ticket Timestamp Column
# ============================================================

filtered_ticket_df = (
    filtered_ticket_df
    .withColumn(
        "EVENT_START_TIME",
        F.to_timestamp(F.col("EVENT_START_TIME"), "MM/dd/yyyy HH:mm:ss")
    )
)


In [173]:
print(alarm_filtered.count())

46925


In [174]:
print(alarm_filtered.columns)

['SITEID', 'RECTIMESTAMP', 'FIRSTOCCURRENCE', 'IDENTIFIER', 'LASTOCCURRENCE', 'SERVICE', 'EMS_NAME', 'NODESTATUS', 'SERVICEAFFECTING', 'CLEARTIME', 'OBJECT', 'NODETYPE', 'SEVERITY', 'EXTENDEDATTR', 'ELEMENTMANAGERIP', 'EVENTID', 'ADDITIONALSTRING', 'ALERTKEY', 'SITENAME', 'ORIGINALSEVERITY', 'SITESTATUS', 'CLASS', 'OBJECTSTATUS', 'CLEAREDBY', 'X733SPECIFICPROB', 'MANAGER', 'NODEALIAS', 'SUMMARY', 'NODE', 'TT_FLAG', 'SERVERSERIAL', 'TMTRANSSTATE', 'TTCREATIONTIME', 'TMTRANSBEGINTIME', 'TT_ID', 'RECEIVEDATPROBE', 'physicalCard', 'OP_TYPE', 'OP_TIME_DWH', 'CIRCLE', 'REGION', 'LOCATION', 'BACKHAULTYPE', 'CONTROLNE', 'ALARM_DESCRIPTION', 'Technology']


In [82]:
alarm_filtered.select("RECEIVEDATPROBE").distinct().show(truncate=False)

+-------------------+
|RECEIVEDATPROBE    |
+-------------------+
|2026-06-30 21:04:54|
|2026-07-09 08:26:05|
|2026-06-28 07:03:11|
|2026-07-04 20:33:58|
|2026-07-09 04:40:15|
|2026-07-09 18:19:24|
|2026-07-06 08:50:09|
|2026-07-15 03:35:54|
|2026-07-19 21:26:19|
|2026-06-29 08:49:38|
|2026-07-01 19:34:05|
|2026-07-01 17:44:05|
|2026-07-03 08:02:59|
|2026-07-22 03:46:47|
|2026-07-19 01:19:16|
|2026-06-26 04:19:41|
|2026-07-19 14:34:31|
|2026-07-05 05:12:50|
|2026-07-05 04:20:57|
|2026-06-27 16:49:40|
+-------------------+
only showing top 20 rows



In [83]:
from pyspark.sql import functions as F

# ============================================================
# STEP 1: Merge alarm with ticket data
# ============================================================

merged = (
    alarm_filtered
    .join(
        filtered_ticket_df.select(
            "HPD_CI",
            "EVENT_START_TIME",
            "INCIDENT_NUMBER"
        ),
        alarm_filtered["SITEID"] == filtered_ticket_df["HPD_CI"],
        how="inner"
    )
)

# ============================================================
# STEP 2: Keep alarms before ticket event
# ============================================================

alarms_before_event = (
    merged
    .filter(
        (F.dayofmonth("LASTOCCURRENCE") <= F.dayofmonth("e"))
        &
        (F.hour("LASTOCCURRENCE") <= F.hour("EVENT_START_TIME"))
    )
)


In [137]:
# Convert to Pandas
alarms_before_event_pd = alarms_before_event.toPandas()

# Save to CSV   
output_path = "/persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month/alarms_before_event.csv"
alarms_before_event_pd.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(alarms_before_event_pd.shape)

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning:

Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning:

Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning:

Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning:

Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning:

Passing

Saved to: /persistent/RAN_fault_v1/Manoj_Code/ran_eda/ran-1month/alarms_before_event.csv
(1537, 50)


In [96]:
print(f"Rows    : {merged.count():,}")
print(f"Columns : {len(merged.columns)}")

Rows    : 79,644
Columns : 49


In [84]:
merged.show(5, truncate=False)

+-------+-------------------+-------------------+----------------------------------------------------------------+-------------------+-----------------------+-----------------------+----------+----------------+-----------------------+------+--------------------+--------+------------+----------------+----------+-------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+--------+----------------+----------+-----+------------+------------------------+--------------------------+--------------------------------------+--------------+--------------------------+--------------+-------+------------+------------+-----------------------+-----------------------+-----+-------------------+------------+-------+------------------------------+-------------+------+--------+------------+---------+--------------------------+----------+-------+-------------------+---------------+
|

In [85]:
from pyspark.sql import functions as F

alarms_before_event.groupBy("RECEIVEDATPROBE") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+-------------------+-----+
|RECEIVEDATPROBE    |count|
+-------------------+-----+
|2026-07-05 02:47:16|36   |
|2026-07-12 04:37:25|34   |
|2026-07-04 01:57:14|31   |
|2026-07-01 02:48:32|29   |
|2026-07-06 03:13:55|28   |
|2026-07-10 03:10:43|27   |
|2026-07-01 07:35:02|23   |
|2026-07-03 04:10:52|22   |
|2026-07-04 05:22:16|21   |
|2026-07-03 04:41:30|21   |
|2026-07-02 03:21:06|21   |
|2026-07-02 04:06:57|21   |
|2026-07-04 03:12:46|21   |
|2026-07-03 03:20:28|21   |
|2026-07-01 05:21:25|21   |
|2026-07-03 05:06:13|21   |
|2026-07-03 06:35:19|21   |
|2026-07-03 02:23:28|21   |
|2026-07-03 06:06:47|21   |
|2026-07-01 06:31:45|21   |
+-------------------+-----+
only showing top 20 rows



In [97]:
print(f"Rows    : {alarms_before_event.count():,}")
print(f"Columns : {len(alarms_before_event.columns)}")

Rows    : 1,537
Columns : 50


In [98]:
from pyspark.sql import functions as F

# ============================================================
# STEP 4: Calculate time gap (hours)
# ============================================================

alarms_before_event = alarms_before_event.withColumn(
    "time_gap_hours",
    (
        F.col("EVENT_START_TIME").cast("long") -
        F.col("RECEIVEDATPROBE").cast("long")
    ) / F.lit(3600.0)
)

# ============================================================
# STEP 5: Keep alarms within last 15 hours
# ============================================================

LOOKBACK_HOURS = 15

alarms_before_event = (
    alarms_before_event
    .filter(
        (F.col("time_gap_hours") >= 0) &
        (F.col("time_gap_hours") <= LOOKBACK_HOURS)
    )
)

In [99]:
alarms_before_event.show(3, truncate=False)

+-------+-------------------+-------------------+---------------------------------------------------------------+-------------------+--------------------+--------------------------------+----------+----------------+-----------------------+------+--------------------+--------+------------+----------------+----------+-----------------------------------------------+---------------------------------------------------------------------------------------------------------------+--------------+----------------+----------+-----+------------+------------------------+------------------------------------+-------------------------------------+--------------+------------------------------------+--------------+-------+------------+------------+-----------------------+-----------------------+-----+-------------------+------------+-------+-----------------------------+---------+------+--------+------------+---------+------------------------------------+----------+-------+-------------------+----------

In [100]:
from pyspark.sql import functions as F

# ── 1. Alarm count per site + incident ──────────────────────────
alarm_count_per_site = (
    alarms_before_event
    .groupBy("SITEID", "INCIDENT_NUMBER")
    .agg(F.count("*").alias("alarm_count"))
    .orderBy(F.col("alarm_count").desc())
)

alarm_count_per_site.show(50, truncate=False)

# ── 2. Distribution across hourly buckets ───────────────────────
distribution = (
    alarms_before_event
    .withColumn(
        "hour_bucket",
        F.concat(
            F.floor(F.col("time_gap_hours")).cast("int").cast("string"),
            F.lit("h - "),
            (F.floor(F.col("time_gap_hours")).cast("int") + 1).cast("string"),
            F.lit("h"),
        )
    )
    .withColumn("bucket_order", F.floor(F.col("time_gap_hours")).cast("int"))
    .groupBy("bucket_order", "hour_bucket")
    .agg(F.count("*").alias("alarm_count"))
    .orderBy("bucket_order")
    .drop("bucket_order")
)

distribution.show(truncate=False)

# ── 3. Summary stats on time_gap_hours ──────────────────────────
alarms_before_event.select(
    F.count("time_gap_hours").alias("total_alarms"),
    F.round(F.mean("time_gap_hours"), 2).alias("mean_hours"),
    F.round(F.stddev("time_gap_hours"), 2).alias("stddev_hours"),
    F.round(F.percentile_approx("time_gap_hours", 0.25), 2).alias("p25"),
    F.round(F.percentile_approx("time_gap_hours", 0.50), 2).alias("median"),
    F.round(F.percentile_approx("time_gap_hours", 0.75), 2).alias("p75"),
    F.round(F.min("time_gap_hours"), 2).alias("min_hours"),
    F.round(F.max("time_gap_hours"), 2).alias("max_hours"),
).show()


+-------+---------------+-----------+
|SITEID |INCIDENT_NUMBER|alarm_count|
+-------+---------------+-----------+
|CMI7624|INC000102025586|218        |
|PCB0168|INC000101832973|137        |
|CMI7624|INC000102080810|112        |
|CMI7624|INC000102283811|108        |
|NKT0147|INC000102071131|104        |
|CMI7624|INC000102302372|102        |
|CMI7624|INC000102345368|91         |
|CMI8863|INC000102085049|66         |
|NKW8534|INC000102253859|38         |
|RCB1003|INC000102096757|37         |
|SPB0164|INC000101858450|34         |
|NKW0330|INC000101865415|29         |
|CMI7624|INC000102189629|28         |
|KCN7918|INC000102072365|27         |
|NKT8583|INC000102361242|25         |
|CMI7624|INC000102186599|25         |
|CRI7175|INC000101910192|22         |
|TAK6704|INC000102040731|17         |
|TAK6710|INC000101857818|16         |
|LPN7167|INC000101879214|13         |
|SPB8540|INC000102094051|12         |
|UTR6727|INC000102297647|11         |
|LPN1909|INC000101919101|11         |
|TAK7611|INC

+-----------+-----------+
|hour_bucket|alarm_count|
+-----------+-----------+
|0h - 1h    |365        |
|1h - 2h    |205        |
|2h - 3h    |130        |
|3h - 4h    |125        |
|4h - 5h    |100        |
|5h - 6h    |81         |
|6h - 7h    |75         |
|7h - 8h    |71         |
|8h - 9h    |136        |
|9h - 10h   |122        |
|10h - 11h  |81         |
|11h - 12h  |27         |
|12h - 13h  |14         |
|13h - 14h  |5          |
+-----------+-----------+



+------------+----------+------------+----+------+---+---------+---------+
|total_alarms|mean_hours|stddev_hours| p25|median|p75|min_hours|max_hours|
+------------+----------+------------+----+------+---+---------+---------+
|        1537|      4.47|        3.64|1.06|  3.44|8.0|      0.0|    13.99|
+------------+----------+------------+----+------+---+---------+---------+



In [187]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.window import Window


# ============================================================
# Remove consecutive duplicates
# ============================================================

def dedup_adjacent(lst):
    if lst is None:
        return []
    
    result = []
    for v in lst:
        if len(result) == 0 or v != result[-1]:
            result.append(v)
    return result


dedup_adjacent_udf = F.udf(
    dedup_adjacent,
    ArrayType(StringType())
)


# ============================================================
# STEP 1: Sort alarms by time
# ============================================================

window_order = (
    Window
    .partitionBy(
        "SITEID",
        "Technology",
        "EVENT_START_TIME",
        "INCIDENT_NUMBER"
    )
    .orderBy(
        "RECEIVEDATPROBE",
        "RECTIMESTAMP"
    )
)


# ============================================================
# STEP 2: Collect ordered alarm sequences
# ============================================================

alarm_sequences = (
    alarms_before_event
    .withColumn(
        "alarm_sequence_raw",
        F.collect_list("X733SPECIFICPROB").over(window_order)
    )
    .withColumn(
        "description_sequence_raw",
        F.collect_list("ALARM_DESCRIPTION").over(window_order)
    )
    .withColumn(
        "additionalstring_sequence_raw",
        F.collect_list("ADDITIONALSTRING").over(window_order)
    )
)


# ============================================================
# STEP 3: Keep one row per ticket event
# ============================================================

alarm_sequences = (
    alarm_sequences
    .groupBy(
        "SITEID",
        "Technology",
        "EVENT_START_TIME",
        "INCIDENT_NUMBER"
    )
    .agg(
        F.max("alarm_sequence_raw").alias("alarm_sequence_raw"),
        F.max("description_sequence_raw").alias("description_sequence_raw"),
        F.max("additionalstring_sequence_raw").alias("additionalstring_sequence_raw")
    )
)


# ============================================================
# STEP 4: Remove consecutive duplicates
# ============================================================

alarm_sequences = (
    alarm_sequences
    .withColumn(
        "alarm_sequence",
        dedup_adjacent_udf(
            "alarm_sequence_raw"
        )
    )
    .withColumn(
        "unique_description",
        dedup_adjacent_udf(
            "description_sequence_raw"
        )
    )
    .withColumn(
        "alarm_summary",
        F.array_distinct(
            F.expr(
                "filter(additionalstring_sequence_raw, x -> x is not null)"
            )
        )
    )
    .drop(
        "alarm_sequence_raw",
        "description_sequence_raw",
        "additionalstring_sequence_raw"
    )
)

In [273]:
alarm_sequences.show(5, truncate=False)

AttributeError: 'DataFrame' object has no attribute 'show'

In [107]:
# Convert Spark DataFrames to Pandas
alarm_filtered_pd = alarm_filtered.toPandas()
filtered_ticket_df_pd = filtered_ticket_df.toPandas()

print("alarm_filtered_pd shape:", alarm_filtered_pd.shape)
print("filtered_ticket_df_pd shape:", filtered_ticket_df_pd.shape)

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' 

alarm_filtered_pd shape: (46826, 46)
filtered_ticket_df_pd shape: (143, 35)


/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' 

In [108]:
print(type(alarm_filtered_pd))
print(type(filtered_ticket_df_pd))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [109]:
import re
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
LOOKBACK_HOURS = 15
TFIDF_MAX_FEATURES = 300
TFIDF_MIN_DF = 2

alarm_to_category = {
    "hardware": [
        "CELL FAULTY", "CELL OPERATION DEGRADED",
        "ESS Service Unavailable", "Service Degraded", "HW Partial Fault", "HW Fault",
        "FRU General Problem", "Linearization Disturbance Performance Degraded",
        "RET Failure", "RSSI Over Threshold", "VSWR Over Threshold", "No Connection",
        "Resource Activation Timeout", "BSC Connection Failure",
    ],
    "infra": [
        "Input Power Failure", "External Alarm Main AC", "External Alarm Main AC Alarm",
        "External Alarm Main AC Power Failure", "External Alarm Main AC Power Alarm",
        "External Alarm LV alarm", "External Alarm Rectifier Failure",
        "External Alarm Rectifier Alarm", "External Alarm Rectifier Urgent Alarm",
        "External Alarm Rectifier Non Urgent Alarm", "External Alarm Rectifier Fail Alarm",
        "External Alarm High Temp Alarm", "Temperature High",
        "Critical Temperature Taken Out of Service", "External Alarm Smoke alarm",
        "External Alarm Controller Failure", "Node Group Sync Loss of All SoCC",
        "PLMN Service Redundancy Lost", "XnC External Link to GNodeB Failure",
        "External Link to GNodeB Failure", "Current Too High",
    ],
    "battery": [
        "External Alarm Battery Failure", "External Alarm Battery Alarm",
        "External Alarm Battery / CB Fail Alarm", "External Alarm Battery Stolen",
    ],
    "suspected_transmission_due_to_power": [
        "S1 Interface Failure", "S1 Link Failure", "NG Interface Failure",
    ],
    "gps": ["Sync Time and Phase Accuracy Too Low", "TimeSyncIO Reference Failed"],
    "txn": [
        "Link Failure", "Link Degraded", "Ethernet Link Failure", "External Link Failure",
        "External Link to GNodeB Failure", "XnC Link to GNodeB Failure",
        "Heartbeat Failure", "Communication Fault", "LOS on SMOD-1, EIF1",
        "NOKIA_SITE_DOWN_IPRAN_NODE_DOWN_CRQ_NMX", "NE and OSS alarms are not in sync",
        "Calendar Clock All NTP Servers Unavailable", "BASE STATION NOTIFICATION",
        "NE3SWS AGENT NOT RESPONDING TO REQUESTS",
        "BASE STATION CONNECTIVITY DEGRADED", "BASE STATION CONNECTIVITY LOST",
    ],
    "door_open_issue": ["External Alarm Door alarm", "External Alarm Door Alarm"],
    "ran": [
        "Carrier Resource Allocation Failure", "Resource Allocation Failure Service Degraded",
        "PLMN Service Degraded", "Suspected Sleeping Cell", "Sleeping Cell(Low Traffic)",
        "LTE Zero Data Volume (12 Hours)", "QOS_LTEHighSessionDrop_HotSpot",
    ],
    "software": ["SW Error"],
    "others": [
        "ALARM DATABASE UPLOAD IN PROGRESS", "Alarm Database upload Failure", "Service Unavailable",
    ],
}

alarm_category_map = {alarm: cat for cat, alarms in alarm_to_category.items() for alarm in alarms}


# ─────────────────────────────────────────────
# STEP 0: Deduplicate alarm_filtered
# ─────────────────────────────────────────────
def dedup_alarm_filtered(alarm_filtered: pd.DataFrame) -> pd.DataFrame:
    """
    Keeps the record with the latest LASTOCCURRENCE per group of identifying columns.
    """
    group_cols = [
        "SITEID", "Technology", "RECTIMESTAMP", "RECEIVEDATPROBE", "CLEARTIME", "X733SPECIFICPROB",
    ]
    df = alarm_filtered.copy()
    df["LASTOCCURRENCE"] = pd.to_datetime(df["LASTOCCURRENCE"], errors="coerce")
    idx = df.groupby(group_cols)["LASTOCCURRENCE"].idxmax()
    return df.loc[idx].reset_index(drop=True)


# ─────────────────────────────────────────────
# STEP 1: Add & clean ALARM_DESCRIPTION column
# ─────────────────────────────────────────────
def _clean_alarm_description(text) -> str:
    """Cleans telecom alarm descriptions by removing dynamic metadata."""
    if pd.isna(text):
        return text
    text = str(text).lower()
    text = re.sub(r'"[^"]*"', ' ', text)
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', ' ', text)
    text = re.sub(r'/[^;| ]*', ' ', text)
    text = re.sub(r'\b[a-zA-Z_]+\s*=\s*[^;|]*', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'\b[a-zA-Z]*\d+[a-zA-Z0-9_-]*\b', ' ', text)
    text = re.sub(r'[\[\]\(\)\{\}]', ' ', text)
    for word in [
        "srcipaddr", "dstipaddr", "suppldstnipaddr", "supplalarminfo",
        "serial_no", "unitname", "additionalfaultid", "path",
        "timeout connecting to", "timeout connecting", "timeout",
        "plmn", "lnbts_parent", "lnbts", "lncel",
    ]:
        text = text.replace(word, " ")
    text = re.sub(r'\|+', '|', text)
    text = re.sub(r';+', ';', text)
    text = re.sub(r'\s*\|\s*', ' | ', text)
    text = re.sub(r'\s*;\s*', '; ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if "|" in text:
        left, right = text.split("|", 1)
        left, right = left.strip(), right.strip()
        if right.startswith(left):
            right = right[len(left):].strip(" ;")
        text = (left + "; " + right) if right else left
    text = re.sub(r'(;\s*){2,}', '; ', text)
    text = re.sub(r'(\|\s*){2,}', '| ', text)
    text = re.sub(r';\s*;', '; ', text)
    text = re.sub(r'\|\s*\|', '| ', text)
    text = re.sub(r';\s*$', '', text)
    text = re.sub(r'\|\s*$', '', text)
    return re.sub(r'\s+', ' ', text).strip(" ;|")


def add_alarm_description(alarm_filtered: pd.DataFrame) -> pd.DataFrame:
    """
    Concatenates X733SPECIFICPROB and SUMMARY into ALARM_DESCRIPTION, then cleans it.
    """
    df = alarm_filtered.copy()
    df["ALARM_DESCRIPTION"] = (
        df["X733SPECIFICPROB"].fillna("").astype(str)
        + " | "
        + df["SUMMARY"].fillna("").astype(str)
    )
    df["ALARM_DESCRIPTION"] = df["ALARM_DESCRIPTION"].apply(_clean_alarm_description)
    return df


# ─────────────────────────────────────────────
# STEP 2: Merge & filter alarms before event
# ─────────────────────────────────────────────
def _ensure_utc(series: pd.Series) -> pd.Series:
    series = pd.to_datetime(series, errors="coerce")
    if series.dt.tz is None:
        return series.dt.tz_localize("UTC")
    return series.dt.tz_convert("UTC")


def build_alarms_before_event(
    alarm_filtered: pd.DataFrame,
    filtered_ticket_df: pd.DataFrame,
    lookback_hours: int = LOOKBACK_HOURS,
) -> pd.DataFrame:
    """
    Merges alarm and ticket data, keeps only alarms that occurred within
    `lookback_hours` before each ticket's EVENT_START_TIME.
    """
    alarm_filtered = alarm_filtered.copy()
    filtered_ticket_df = filtered_ticket_df.copy()

    alarm_filtered["LASTOCCURRENCE"] = _ensure_utc(alarm_filtered["LASTOCCURRENCE"])
    alarm_filtered["RECTIMESTAMP"] = _ensure_utc(alarm_filtered["RECTIMESTAMP"])
    alarm_filtered["RECEIVEDATPROBE"] = _ensure_utc(alarm_filtered["RECEIVEDATPROBE"])
    filtered_ticket_df["EVENT_START_TIME"] = _ensure_utc(filtered_ticket_df["EVENT_START_TIME"])

    merged = alarm_filtered.merge(
        filtered_ticket_df[["HPD_CI", "EVENT_START_TIME","INCIDENT_NUMBER"]],
        left_on="SITEID",
        right_on="HPD_CI",
        how="inner",
    )

    alarms_before = merged[
        (merged["LASTOCCURRENCE"].dt.day <= merged["EVENT_START_TIME"].dt.day) &
        (merged["LASTOCCURRENCE"].dt.hour <= merged["EVENT_START_TIME"].dt.hour)
    ].copy()

    alarms_before["time_gap_hours"] = (
        alarms_before["EVENT_START_TIME"] - alarms_before["RECEIVEDATPROBE"]
    ).dt.total_seconds() / 3600

    alarms_before = alarms_before[
        (alarms_before["time_gap_hours"] >= 0) &
        (alarms_before["time_gap_hours"] <= lookback_hours)
    ].copy()

    return alarms_before


# ─────────────────────────────────────────────
# SEVERITY ENCODING
# ─────────────────────────────────────────────
SEVERITY_INVERT = {0: 5, 1: 4, 2: 3, 3: 2, 4: 1, 5: 0}


def _encode_severity(s) -> int:
    """Converts raw severity (0=Critical..5=Clear) to inverted scale (5=Critical..0=Clear)."""
    if pd.isna(s):
        return 0
    try:
        return SEVERITY_INVERT.get(int(float(s)), 0)
    except (ValueError, TypeError):
        return 0


# ─────────────────────────────────────────────
# STEP 3: Build alarm sequences per site/event
# ─────────────────────────────────────────────
def _dedup_adjacent(lst: list) -> list:
    """Remove consecutive duplicate values while preserving order."""
    return [v for i, v in enumerate(lst) if i == 0 or v != lst[i - 1]]


def build_alarm_sequences(alarms_before_event: pd.DataFrame) -> pd.DataFrame:
    """
    Groups alarms by site + event.
    Extracts rich aggregations from raw row-level columns BEFORE they are lost.
    """
    df = alarms_before_event.copy()
    df = df.sort_values(["RECEIVEDATPROBE", "RECTIMESTAMP"])

    df["_sev_num"] = df["SEVERITY"].apply(_encode_severity)
    df["_orig_sev_num"] = df["ORIGINALSEVERITY"].apply(_encode_severity)
    df["_alarm_duration_h"] = df["time_gap_hours"]
    df["_is_uncleared"] = df["CLEARTIME"].isna().astype(int)
    df["_is_service_affecting"] = pd.to_numeric(df["SERVICEAFFECTING"], errors="coerce").fillna(0).clip(0, 1).astype(int)
    df["_has_tt"] = (pd.to_numeric(df["TT_FLAG"], errors="coerce").fillna(0) > 0).astype(int)

    groupby_keys = ["SITEID", "Technology", "EVENT_START_TIME","INCIDENT_NUMBER"]

    alarm_sequences = (
        df.groupby(groupby_keys)
        .agg(
            # ── sequence / text ──────────────────────────────────────────
            alarm_sequence=("X733SPECIFICPROB", lambda x: _dedup_adjacent(list(x))),
            unique_description=("ALARM_DESCRIPTION", lambda x: _dedup_adjacent(list(pd.unique(x)))),
            alarm_summary=("ADDITIONALSTRING", lambda x: list(pd.unique(x.dropna()))),

            # ── severity ─────────────────────────────────────────────────
            max_severity=("_sev_num", "max"),
            mean_severity=("_sev_num", "mean"),
            critical_alarm_count=("_sev_num", lambda x: (x == 5).sum()),
            major_alarm_count=("_sev_num", lambda x: (x == 4).sum()),
            severity_escalation=(
                "_sev_num",
                lambda x: int(any(o < s for o, s in zip(df.loc[x.index, "_orig_sev_num"], x))),
            ),

            # ── service impact ───────────────────────────────────────────
            service_affecting_count=("_is_service_affecting", "sum"),
            service_affecting_ratio=("_is_service_affecting", "mean"),

            # ── temporal ─────────────────────────────────────────────────
            alarm_duration_span_hours=(
                "RECEIVEDATPROBE",
                lambda x: (
                    pd.to_datetime(x).max() - pd.to_datetime(x).min()
                ).total_seconds() / 3600,
            ),
            avg_alarm_duration_hours=("_alarm_duration_h", "mean"),
            max_alarm_duration_hours=("_alarm_duration_h", "max"),

            # ── clearance / recovery ─────────────────────────────────────
            uncleared_alarm_count=("_is_uncleared", "sum"),
            cleared_alarm_ratio=("_is_uncleared", lambda x: 1 - x.mean()),

            # ── trouble ticket linkage ───────────────────────────────────
            tt_linked_alarm_count=("_has_tt", "sum"),

            # ── infra context ────────────────────────────────────────────
            backhaul_type=("BACKHAULTYPE", lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown")
        )
        .reset_index()
    )

    last_occurrence = (
        df.groupby(groupby_keys)["LASTOCCURRENCE"]
        .max()
        .reset_index()
        .rename(columns={"LASTOCCURRENCE": "_last_occ"})
    )
    alarm_sequences = alarm_sequences.merge(last_occurrence, on=groupby_keys, how="left")
    alarm_sequences["alarm_recency_hours"] = (
        alarm_sequences["EVENT_START_TIME"] - pd.to_datetime(alarm_sequences["_last_occ"])
    ).dt.total_seconds() / 3600
    alarm_sequences.drop(columns=["_last_occ"], inplace=True)

    return alarm_sequences


# ─────────────────────────────────────────────
# STEP 4: Alarm frequency & category mapping
# ─────────────────────────────────────────────
def get_alarm_frequency_with_categories(alarm_sequences: pd.DataFrame) -> pd.DataFrame:
    """
    Counts alarm frequency across all sequences and maps each alarm to its category.
    """
    all_alarms = [alarm for seq in alarm_sequences["alarm_sequence"] for alarm in seq]
    alarm_counts = (
        pd.DataFrame(Counter(all_alarms).items(), columns=["alarm", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    alarm_counts["category"] = alarm_counts["alarm"].map(alarm_category_map).fillna("unknown")
    return alarm_counts


# ─────────────────────────────────────────────
# STEP 5: Clean description for TF-IDF
# ─────────────────────────────────────────────
def clean_description(desc) -> str:
    """
    Normalizes alarm description text into space-separated underscore tokens
    suitable for TF-IDF ingestion.
    """
    if desc is None or (isinstance(desc, float) and pd.isna(desc)):
        return ""
    text = " | ".join(map(str, desc)) if isinstance(desc, list) else str(desc)
    cleaned = []
    for alarm in text.split("|"):
        alarm = alarm.lower().strip()
        alarm = re.sub(r"[^a-z0-9\s]", " ", alarm)
        alarm = re.sub(r"\s+", " ", alarm).strip()
        alarm = alarm.replace(" ", "_")
        if alarm:
            cleaned.append(alarm)
    return " ".join(cleaned)


# ─────────────────────────────────────────────
# STEP 6: TF-IDF vectorization
# ─────────────────────────────────────────────
def build_tfidf_features(
    alarm_sequences: pd.DataFrame,
    max_features: int = TFIDF_MAX_FEATURES,
    min_df: int = TFIDF_MIN_DF,
) -> tuple[pd.DataFrame, TfidfVectorizer]:
    """
    Cleans unique_description and applies TF-IDF.

    Returns:
        tfidf_df   : DataFrame of TF-IDF features aligned to alarm_sequences index
        vectorizer : fitted TfidfVectorizer (for inference / persistence)
    """
    alarm_sequences = alarm_sequences.copy()
    alarm_sequences["clean_description"] = alarm_sequences["unique_description"].apply(clean_description)

    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
    X_tfidf = vectorizer.fit_transform(alarm_sequences["clean_description"])

    tfidf_df = pd.DataFrame(
        X_tfidf.toarray(),
        columns=vectorizer.get_feature_names_out(),
        index=alarm_sequences.index,
    )
    return tfidf_df, vectorizer


# ─────────────────────────────────────────────
# STEP 7: Sequence-level engineered features
# ─────────────────────────────────────────────
def build_sequence_features(alarm_sequences: pd.DataFrame) -> pd.DataFrame:
    """
    Derives features purely from the alarm_sequence list (post-groupby).
    """
    df = alarm_sequences.copy()

    df["alarm_count"] = df["alarm_sequence"].apply(len)
    df["unique_alarm_count"] = df["alarm_sequence"].apply(lambda x: len(set(x)))
    df["alarm_diversity"] = df.apply(
        lambda r: r["unique_alarm_count"] / r["alarm_count"] if r["alarm_count"] > 0 else 0,
        axis=1,
    )

    def top_alarm_stats(seq):
        if not seq:
            return pd.Series({"top_alarm": None, "top_alarm_freq": 0, "top_alarm_ratio": 0.0})
        c = Counter(seq)
        top, freq = c.most_common(1)[0]
        return pd.Series({"top_alarm": top, "top_alarm_freq": freq, "top_alarm_ratio": freq / len(seq)})

    df = df.join(df["alarm_sequence"].apply(top_alarm_stats))

    category_flags = {
        "has_infra_alarm": "infra",
        "has_hardware_alarm": "hardware",
        "has_txn_alarm": "txn",
        "has_battery_alarm": "battery",
        "has_power_alarm": "suspected_transmission_due_to_power",
        "has_gps_alarm": "gps",
        "has_ran_alarm": "ran",
    }
    for col, cat in category_flags.items():
        df[col] = df["alarm_sequence"].apply(
            lambda seq: int(any(alarm_category_map.get(a) == cat for a in seq))
        )

    df["category_count"] = df["alarm_sequence"].apply(
        lambda seq: len({alarm_category_map.get(a, "unknown") for a in seq})
    )
    df["repeat_alarm_count"] = df["alarm_sequence"].apply(
        lambda seq: sum(1 for v in Counter(seq).values() if v > 1)
    )
    df["alarm_burst_rate"] = df.apply(
        lambda r: r["alarm_count"] / r["alarm_duration_span_hours"]
        if r.get("alarm_duration_span_hours", 0) > 0
        else r["alarm_count"],
        axis=1,
    )

    return df


# ─────────────────────────────────────────────
# MASTER PIPELINE
# ─────────────────────────────────────────────
def build_alarm_feature_set(
    alarm_filtered: pd.DataFrame,
    filtered_ticket_df: pd.DataFrame,
    lookback_hours: int = LOOKBACK_HOURS,
    tfidf_max_features: int = TFIDF_MAX_FEATURES,
    tfidf_min_df: int = TFIDF_MIN_DF,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, TfidfVectorizer]:
    """
    End-to-end pipeline that returns:
        alarm_sequences  : base sequence DataFrame with engineered features
        tfidf_df         : TF-IDF feature matrix (aligned by index)
        alarm_counts     : alarm frequency + category table
        vectorizer       : fitted TfidfVectorizer
    """
    alarm_filtered = dedup_alarm_filtered(alarm_filtered)
    alarm_filtered = add_alarm_description(alarm_filtered)
    alarms_before_event = build_alarms_before_event(alarm_filtered, filtered_ticket_df, lookback_hours)
    alarm_sequences = build_alarm_sequences(alarms_before_event)
    alarm_sequences = build_sequence_features(alarm_sequences)
    alarm_counts = get_alarm_frequency_with_categories(alarm_sequences)
    tfidf_df, vectorizer = build_tfidf_features(alarm_sequences, tfidf_max_features, tfidf_min_df)

    print(f"[alarm_features] sequences: {len(alarm_sequences)} | tfidf shape: {tfidf_df.shape}")
    print(alarm_counts.head(20).to_string(index=False))

    return alarm_sequences, tfidf_df, alarm_counts, vectorizer


# ─────────────────────────────────────────────
# USAGE EXAMPLE
# ─────────────────────────────────────────────
alarm_sequences, tfidf_df, alarm_counts, vectorizer = build_alarm_feature_set(
    alarm_filtered_pd, filtered_ticket_df_pd
)

# Combine for modelling
model_df = pd.concat([alarm_sequences.reset_index(drop=True),
                       tfidf_df.reset_index(drop=True)], axis=1)


[alarm_features] sequences: 77 | tfidf shape: (77, 22)
                                    alarm  count category
        NE and OSS alarms are not in sync    174      txn
                            No Connection    141 hardware
               XnC Link to GNodeB Failure     53      txn
                    External Link Failure     39      txn
                    Ethernet Link Failure     37      txn
          External Link to GNodeB Failure     18      txn
                         Temperature High     15    infra
     Sync Time and Phase Accuracy Too Low     14      gps
                        Heartbeat Failure     13      txn
     External Alarm Main AC Power Failure     11    infra
    External Alarm Rectifier Urgent Alarm     10    infra
                      Input Power Failure      8    infra
                  External Alarm LV alarm      8    infra
             PLMN Service Redundancy Lost      7    infra
                                 HW Fault      7 hardware
             Exte

In [284]:
# model_df[model_df["NR_Cell_Availability"] == 0]

In [283]:
# model_df.columns

In [272]:
# model_df.duplicated().sum()

In [ ]:
#####creating the KPI DATA For the analysis####

In [112]:
# ============================================================
# PRE-OUTAGE KPI ANALYSIS
# ============================================================
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
 
 
# ============================================================
# CONFIGURATION
# ============================================================
 
# selected_site = "AYT0381P_9NB01"
# cell_id = "AYT0381H_7NB01_S01"
 
target = "NR_Cell_Availability"
 
# How much history you want before outage
HOURS_BEFORE_OUTAGE = 4
 
 
# ============================================================
# 1. FILTER SITE + CELL
# ============================================================
 
# cell_df = (
#     df
#     .filter(
#         (F.col("SITE") == selected_site)
# &
#         (F.col("CELLID") == cell_id)
#     )
# )
 
cell_df = filtered_df
 
# ============================================================
# 2. CONVERT COLLECTTIME
# ============================================================
 
# Your COLLECTTIME:
# 20260701143000
#
# Format:
# yyyyMMddHHmmss
 
# ============================================================
# 2. CONVERT COLLECTTIME
# ============================================================

cell_df = cell_df.withColumn(
    "COLLECTTIME_TS",
    F.to_timestamp(
        F.col("COLLECTTIME"),
        "yyyyMMddHHmmss"
    )
)
 
# ============================================================
# 3. SORT CELL DATA
# ============================================================
 
cell_window = (
    Window
    .partitionBy(
        "SITE",
        "CELLID"
    )
    .orderBy(
        "COLLECTTIME_TS"
    )
)
 
 
# ============================================================
# 4. GET PREVIOUS AVAILABILITY
# ============================================================
 
cell_df = cell_df.withColumn(
    "previous_availability",
    F.lag(
        F.col(target)
    ).over(cell_window)
)
 
 
# ============================================================
# 5. IDENTIFY OUTAGE START
# ============================================================
#
# We DON'T want:
#
# 14:30 -> 0  <- outage starts
# 14:45 -> 0  <- same outage
#
# We only want 14:30.
#
# Therefore:
#
# current availability = 0
# previous availability > 0
#
# ============================================================
 
cell_df = cell_df.withColumn(
    "outage_start",
    F.when(
        (F.col(target) == 0)
&
        (F.col("previous_availability") > 0),
        1
    ).otherwise(0)
)
 
 
# ============================================================
# 6. SEE ALL OUTAGE STARTS
# ============================================================
 
print("Outage starts:")
 
(
    cell_df
    .filter(
        F.col("outage_start") == 1
    )
    .select(
        "SITE",
        "CELLID",
        "COLLECTTIME",
        "COLLECTTIME_TS",
        "previous_availability",
        target
    )
    .orderBy("COLLECTTIME_TS")
    .show(
        100,
        truncate=False
    )
)

Outage starts:


+--------------+------------------+--------------+-------------------+---------------------+--------------------+
|SITE          |CELLID            |COLLECTTIME   |COLLECTTIME_TS     |previous_availability|NR_Cell_Availability|
+--------------+------------------+--------------+-------------------+---------------------+--------------------+
|CMI7624L_6NB02|CMI7624L_68T02_S12|20260623084500|2026-06-23 08:45:00|60.222               |0.0                 |
|CMI7624L_6NB02|CMI7624L_68T02_S11|20260623084500|2026-06-23 08:45:00|60.222               |0.0                 |
|CMI6333L_6NB04|CMI7673L_6MM04_R11|20260623113000|2026-06-23 11:30:00|0.667                |0.0                 |
|CMI6333L_6NB04|CMI7673L_6MM04_R11|20260624041500|2026-06-24 04:15:00|0.778                |0.0                 |
|CMI6333L_6NB04|CMI7673L_6MM04_R11|20260624053000|2026-06-24 05:30:00|8.222                |0.0                 |
|SRBEV83L_6NB02|SRBEV83L_6MM02_S12|20260624084500|2026-06-24 08:45:00|100.0             

In [115]:
# ============================================================
# 7. SELECT OUTAGE
# ============================================================

outage_rows = (
    cell_df
    .filter(
        F.col("outage_start") == 1
    )
    .orderBy("COLLECTTIME_TS")
#     .first()
    .limit(2)
    .collect()
)


if len(outage_rows) < 2:
    raise ValueError(
        f"Less than 2 healthy -> outage transitions found for {cell_id}"
    )
    
outage_row = outage_rows[1]


outage_time = outage_row["COLLECTTIME_TS"]

print(
    "Selected outage time:",
    outage_time
)

Selected outage time: 2026-06-23 08:45:00


In [116]:
# ============================================================
# 8. TAKE ONLY PRE-OUTAGE DATA
# ============================================================

pre_outage_df = (
    cell_df
    .filter(

        (
            F.col("COLLECTTIME_TS")
            >=
            F.expr(
                f"timestamp'{outage_time}' - INTERVAL {HOURS_BEFORE_OUTAGE} HOURS"
            )
        )

        &

        (
            F.col("COLLECTTIME_TS")
            <=
            F.lit(outage_time)
        )
    )
    .orderBy(
        "COLLECTTIME_TS"
    )
)

In [117]:
pre_outage_df.select(
    "SITE",
    "CELLID",
    "COLLECTTIME_TS",
    target
).show(
    100,
    truncate=False
)

+--------------+------------------+-------------------+--------------------+
|SITE          |CELLID            |COLLECTTIME_TS     |NR_Cell_Availability|
+--------------+------------------+-------------------+--------------------+
|AYT6773P_9NB01|AYT6773H_7NB01_S01|2026-06-23 04:45:00|100.0               |
|RCB7173L_6NB06|RCB7173L_68T06_S14|2026-06-23 04:45:00|100.0               |
|RCB1003L_6NB01|RCB1003L_6MM01_S13|2026-06-23 04:45:00|100.0               |
|NKT0310T_2NB01|NKT0310H_7NB01_S03|2026-06-23 04:45:00|100.0               |
|RCB0317T_2NB01|RCB0317H_7NB01_S03|2026-06-23 04:45:00|100.0               |
|SPB0164T_2NB01|SPB0164H_7NB01_S01|2026-06-23 04:45:00|100.0               |
|NKT8583P_9NB01|NKT8583H_7NB01_S01|2026-06-23 04:45:00|100.0               |
|RCBC011T_2NB01|RCBC011H_7NB01_S04|2026-06-23 04:45:00|100.0               |
|NKW1925P_9NB01|NKW1925H_7NB01_S03|2026-06-23 04:45:00|100.0               |
|PCB7676L_6NB03|PCB7676L_6MM03_S12|2026-06-23 04:45:00|100.0               |

In [118]:
# ============================================================
# 9. CONVERT PRE-OUTAGE DATA TO PANDAS
# ============================================================

columns_needed = [
    "SITE",
    "CELLID",
    "COLLECTTIME_TS",
    target,
] 

cell_pd = (
    pre_outage_df
    .select(*columns_needed)
    .toPandas()
)

# ============================================================
# Sort by Site, Cell and Time
# ============================================================

cell_pd = cell_pd.sort_values(
    ["SITE", "CELLID", "COLLECTTIME_TS"]
).reset_index(drop=True)

# ============================================================
# Verify
# ============================================================

print(cell_pd.dtypes)

print(
    cell_pd[
        [
            "SITE",
            "CELLID",
            "COLLECTTIME_TS",
            target,
        ]
    ].head(20)
)

/opt/anaconda/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


SITE                            object
CELLID                          object
COLLECTTIME_TS          datetime64[ns]
NR_Cell_Availability           float64
dtype: object
              SITE              CELLID      COLLECTTIME_TS  \
0   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 04:45:00   
1   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 05:00:00   
2   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 05:15:00   
3   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 05:30:00   
4   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 05:45:00   
5   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 06:00:00   
6   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 06:15:00   
7   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 06:30:00   
8   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 06:45:00   
9   ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 07:00:00   
10  ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 07:15:00   
11  ATG0096T_2NB01  ATG0096H_7NB01_S01 2026-06-23 07:30:00   
12  ATG0096T_2NB01  ATG0

In [119]:
cell_pd["SITEID"] = cell_pd["SITE"].str[:7]

In [121]:
model_df_merged = model_df.merge(
    cell_pd,
    left_on="SITEID",
    right_on="SITEID",
    how="left"
)

In [132]:
# # Save as CSV
# model_df_merged.to_csv("model_df_merged.csv", index=False)

# print("Saved as model_df_merged.csv")

Saved as model_df_merged.csv


In [133]:
# # Save as CSV
model_df.to_csv("model_df_sequence.csv", index=False)

print("Saved as model_df_sequence.csv")

Saved as model_df_sequence.csv


In [ ]:
#######code for plotting the correaltions#####

In [122]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.feature_selection import mutual_info_regression

# ============================================================
# CONFIG
# ============================================================
TARGET = "NR_Cell_Availability"

# ============================================================
# Build plot_df from model_df
# ============================================================
exclude_cols = ["SITEID", "Technology", "EVENT_START_TIME", "HPD_CI",
                "alarm_sequence", "unique_description", "alarm_summary",
                "clean_description", "top_alarm", "first_alarm", "last_alarm",
                "dominant_category", "first_alarm_category", "last_alarm_category",
                "backhaul_type", "circle", "region"]

numeric_features = [
    c for c in model_df_merged.select_dtypes(include=[np.number]).columns
    if c != TARGET and c not in exclude_cols
]

print(f"Number of numeric features: {len(numeric_features)}")

plot_df = model_df_merged[numeric_features + [TARGET]].copy()
plot_df = plot_df[plot_df[TARGET].notna()]

# ============================================================
# 1. Target distribution
# ============================================================
target_counts = plot_df[TARGET].value_counts().sort_index().reset_index()
target_counts.columns = [TARGET, "Count"]
fig = px.bar(target_counts, x=TARGET, y="Count",
             title=f"Distribution of {TARGET}")
fig.write_html("plot_1_target_distribution.html")
print("Saved: plot_1_target_distribution.html")

# ============================================================
# 2. Pearson correlation with target
# ============================================================
correlations = (
    plot_df[numeric_features + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .dropna()
)
correlations = correlations.loc[correlations.abs().sort_values(ascending=False).index]
print("\nTop 30 correlations with target:")
print(correlations.head(30).to_string())

# Sort descending for plotting (highest positive first, most negative last)
corr_sorted = correlations.sort_values(ascending=False)

# ============================================================
# 3. Correlation bar plots (chunked)
# ============================================================
def plot_correlation_range(corr_data, start, end, title, filename):
    subset = corr_data.iloc[start:end]
    if subset.empty:
        return
    colors = ["#d73027" if v < 0 else "#1a9850" for v in subset.values]
    fig = go.Figure(go.Bar(
        x=subset.values,
        y=subset.index,
        orientation="h",
        marker_color=colors,
        text=[f"{v:.3f}" for v in subset.values],
        textposition="outside"
    ))
    fig.update_layout(
        title=title,
        xaxis_title=f"Pearson Correlation with {TARGET}",
        yaxis_title="Feature",
        xaxis=dict(range=[-0.6, 0.6], dtick=0.1),
        height=max(800, len(subset) * 35),
        margin=dict(l=400, r=100, t=60, b=60),
        yaxis=dict(tickfont=dict(size=11))
    )
    fig.add_vline(x=0, line_width=1, line_color="black")
    fig.write_html(filename)
    print(f"Saved: {filename}")

chunk = 60
for i, start in enumerate(range(0, len(corr_sorted), chunk)):
    plot_correlation_range(
        corr_sorted, start, start + chunk,
        title=f"Feature Correlation with {TARGET} (Part {i+1})",
        filename=f"plot_3_correlation_part{i+1}.html"
    )

# ============================================================
# 4a. Compute Mutual Information
# ============================================================
X_mi = plot_df[numeric_features].fillna(0)
y_mi = plot_df[TARGET].fillna(0)

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)

mi_result = (
    pd.DataFrame({"feature": numeric_features, "mutual_information": mi_scores})
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 30 features by Mutual Information:")
print(mi_result.head(30).to_string(index=False))

# ============================================================
# 4b. Summary DataFrame (needed for section 5)
# ============================================================
summary = (
    mi_result.set_index("feature")
    .join(correlations.rename("pearson_correlation"), how="left")
    .assign(abs_correlation=lambda d: d["pearson_correlation"].abs())
    .sort_values("mutual_information", ascending=False)
)

# ============================================================
# 4. Mutual Information plot
# ============================================================
top_mi = mi_result.head(50)
fig = go.Figure(go.Bar(
    x=top_mi["mutual_information"][::-1].values,
    y=top_mi["feature"][::-1].values,
    orientation="h",
    marker_color="#2166ac",
    text=[f"{v:.4f}" for v in top_mi["mutual_information"][::-1].values],
    textposition="outside"
))
fig.update_layout(
    title=f"Top {len(top_mi)} Features by Mutual Information with {TARGET}",
    xaxis_title="Mutual Information Score",
    yaxis_title="Feature",
    height=max(800, len(top_mi) * 35),
    margin=dict(l=400, r=100, t=60, b=60),
    yaxis=dict(tickfont=dict(size=11))
)
fig.write_html("plot_4_mutual_information.html")
print("Saved: plot_4_mutual_information.html")

# ============================================================
# 5. Summary table as HTML
# ============================================================
fig_table = go.Figure(go.Table(
    header=dict(
        values=["Feature", "Mutual Information", "Pearson Correlation", "Abs Correlation"],
        fill_color="#2166ac",
        font=dict(color="white", size=12),
        align="left"
    ),
    cells=dict(
        values=[
            summary.index.tolist(),
            summary["mutual_information"].round(4).tolist(),
            summary["pearson_correlation"].round(4).tolist(),
            summary["abs_correlation"].round(4).tolist()
        ],
        fill_color=[["#f0f0f0" if i % 2 == 0 else "white" for i in range(len(summary))]],
        align="left",
        font=dict(size=11)
    )
))
fig_table.update_layout(
    title="Feature Importance Summary (MI + Correlation)",
    height=max(600, len(summary) * 30)
)
fig_table.write_html("plot_5_summary_table.html")
print("Saved: plot_5_summary_table.html")


Number of numeric features: 51
Saved: plot_1_target_distribution.html

Top 30 correlations with target:
hw_fault                                              -0.515039
top_alarm_freq                                        -0.471370
critical_temperature_taken_out_of_service             -0.466025
critical_alarm_count                                  -0.414290
alarm_count                                           -0.371864
temperature_high                                      -0.357213
vswr_over_threshold                                   -0.329796
no_connection                                         -0.296491
has_hardware_alarm                                    -0.264862
alarm_diversity                                        0.261692
category_count                                        -0.166571
alarm_duration_span_hours                             -0.160260
ne_and_oss_alarms_are_not_in_sync                     -0.132193
max_alarm_duration_hours                              -0.114758


In [ ]:
##### filterin on the basis of MI and abs correlation scores

In [129]:
# ============================================================
# Threshold
# ============================================================

THRESHOLD = 0.01

# ============================================================
# MI Ranking
# ============================================================

mi_rank = (
    mi_result
    .query("mutual_information >= @THRESHOLD")
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# Absolute Correlation Ranking
# ============================================================

corr_rank = (
    summary.reset_index()[["feature", "abs_correlation"]]
    .query("abs_correlation >= @THRESHOLD")
    .sort_values("abs_correlation", ascending=False)
    .reset_index(drop=True)
)

In [130]:
# ============================================================
# 1. MI Selected Features
# ============================================================

mi_selected = (
    mi_rank[["feature", "mutual_information"]]
    .merge(
        summary.reset_index()[["feature", "abs_correlation"]],
        on="feature",
        how="left"
    )
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

print(f"\nMI Selected Features ({len(mi_selected)})")
display(mi_selected)


# ============================================================
# 2. Correlation Selected Features
# ============================================================

corr_selected = (
    corr_rank[["feature", "abs_correlation"]]
    .merge(
        mi_result[["feature", "mutual_information"]],
        on="feature",
        how="left"
    )
    .sort_values("abs_correlation", ascending=False)
    .reset_index(drop=True)
)

print(f"\nCorrelation Selected Features ({len(corr_selected)})")
display(corr_selected)


# ============================================================
# 3. Common Features
# ============================================================

common_selected = pd.merge(
    mi_selected,
    corr_selected,
    on="feature",
    suffixes=("_mi", "_corr")
)

# Keep only one MI and one Abs Correlation column
common_selected = common_selected[
    ["feature", "mutual_information_mi", "abs_correlation_mi"]
].rename(
    columns={
        "mutual_information_mi": "mutual_information",
        "abs_correlation_mi": "abs_correlation"
    }
)

common_selected = common_selected.sort_values(
    "mutual_information",
    ascending=False
).reset_index(drop=True)

print(f"\nCommon Features ({len(common_selected)})")
display(common_selected)


MI Selected Features (29)


,feature,mutual_information,abs_correlation
0,ne_and_oss_alarms_are_not_in_sync,0.061805,0.132193
1,critical_alarm_count,0.061524,0.414290
2,alarm_diversity,0.060479,0.261692
3,top_alarm_freq,0.048269,0.471370
4,temperature_high,0.039390,0.357213
5,no_connection,0.038977,0.296491
6,alarm_count,0.037446,0.371864
7,has_hardware_alarm,0.032826,0.264862
8,top_alarm_ratio,0.031273,0.076539
9,alarm_burst_rate,0.030428,0.022460



Correlation Selected Features (45)


,feature,abs_correlation,mutual_information
0,hw_fault,0.515039,0.013975
1,top_alarm_freq,0.471370,0.048269
2,critical_temperature_taken_out_of_service,0.466025,0.011301
3,critical_alarm_count,0.414290,0.061524
4,alarm_count,0.371864,0.037446
5,temperature_high,0.357213,0.039390
6,vswr_over_threshold,0.329796,0.023217
7,no_connection,0.296491,0.038977
8,has_hardware_alarm,0.264862,0.032826
9,alarm_diversity,0.261692,0.060479



Common Features (27)


,feature,mutual_information,abs_correlation
0,ne_and_oss_alarms_are_not_in_sync,0.061805,0.132193
1,critical_alarm_count,0.061524,0.414290
2,alarm_diversity,0.060479,0.261692
3,top_alarm_freq,0.048269,0.471370
4,temperature_high,0.039390,0.357213
5,no_connection,0.038977,0.296491
6,alarm_count,0.037446,0.371864
7,has_hardware_alarm,0.032826,0.264862
8,top_alarm_ratio,0.031273,0.076539
9,alarm_burst_rate,0.030428,0.022460


In [131]:
# ============================================================
# Keep feature with highest Abs Correlation for each MI value
# ============================================================

MI_DECIMALS = 3    # Round MI to 3 decimal places

mi_filtered = (
    mi_selected.copy()
)

# Create MI groups
mi_filtered["MI_Group"] = mi_filtered["mutual_information"].round(MI_DECIMALS)

# Within each MI group keep the feature with highest abs correlation
best_features = (
    mi_filtered
    .sort_values(
        ["MI_Group", "abs_correlation"],
        ascending=[False, False]
    )
    .groupby("MI_Group", as_index=False)
    .first()
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(best_features[["feature", "mutual_information", "abs_correlation"]])

,feature,mutual_information,abs_correlation
0,critical_alarm_count,0.061524,0.414290
1,alarm_diversity,0.060479,0.261692
2,top_alarm_freq,0.048269,0.471370
3,temperature_high,0.039390,0.357213
4,alarm_count,0.037446,0.371864
5,has_hardware_alarm,0.032826,0.264862
6,top_alarm_ratio,0.031273,0.076539
7,alarm_burst_rate,0.030428,0.022460
8,alarm_duration_span_hours,0.027770,0.160260
9,carrier_resource_allocation_failure,0.025597,0.059571


In [ ]:
#####Now we will filter the data for a particulr site and do the plottings

In [318]:
model_df_merged['SITEID'].value_counts()

CRI6320    255
TAK6710    238
TAK6704    204
AYT8652    153
NKW1925    119
          ... 
NKT0347     51
CRI3188     34
CMI6333     34
AYT7432     34
CRI7175     34
Name: SITEID, Length: 71, dtype: int64

In [329]:
model_df_filtered = model_df_merged[
    (model_df_merged["SITEID"] == "CMI7624") &
    (model_df_merged["CELLID"] == "CMI7624L_68T02_S11")
]

model_df_filtered.head()

,SITEID,Technology,EVENT_START_TIME,INCIDENT_NUMBER,alarm_sequence,unique_description,alarm_summary,max_severity,mean_severity,critical_alarm_count,...,node_group_sync_loss_of_all_socc,service_redundancy_lost,sync_time_and_phase_accuracy_too_low,temperature_high,vswr_over_threshold,xnc_link_to_gnodeb_failure,SITE,CELLID,COLLECTTIME_TS,NR_Cell_Availability
919,CMI7624,5G,2026-07-04 13:58:23+00:00,INC000102025586,"[HW Fault, VSWR Over Threshold, Temperature Hi...","[hw fault, vswr over threshold, temperature hi...","[ReturnLoss 9.4 dB, VSWR 2.0, Sensitivity 50%,...",5,4.197248,159,...,0.0,0.0,0.0,0.393498,0.508662,0.0,CMI7624L_6NB02,CMI7624L_68T02_S11,2026-06-23 08:00:00,65.556
920,CMI7624,5G,2026-07-04 13:58:23+00:00,INC000102025586,"[HW Fault, VSWR Over Threshold, Temperature Hi...","[hw fault, vswr over threshold, temperature hi...","[ReturnLoss 9.4 dB, VSWR 2.0, Sensitivity 50%,...",5,4.197248,159,...,0.0,0.0,0.0,0.393498,0.508662,0.0,CMI7624L_6NB02,CMI7624L_68T02_S11,2026-06-23 08:15:00,92.444
923,CMI7624,5G,2026-07-04 13:58:23+00:00,INC000102025586,"[HW Fault, VSWR Over Threshold, Temperature Hi...","[hw fault, vswr over threshold, temperature hi...","[ReturnLoss 9.4 dB, VSWR 2.0, Sensitivity 50%,...",5,4.197248,159,...,0.0,0.0,0.0,0.393498,0.508662,0.0,CMI7624L_6NB02,CMI7624L_68T02_S11,2026-06-23 08:30:00,60.222
924,CMI7624,5G,2026-07-04 13:58:23+00:00,INC000102025586,"[HW Fault, VSWR Over Threshold, Temperature Hi...","[hw fault, vswr over threshold, temperature hi...","[ReturnLoss 9.4 dB, VSWR 2.0, Sensitivity 50%,...",5,4.197248,159,...,0.0,0.0,0.0,0.393498,0.508662,0.0,CMI7624L_6NB02,CMI7624L_68T02_S11,2026-06-23 08:45:00,0.000
927,CMI7624,5G,2026-07-07 12:46:57+00:00,INC000102080810,"[HW Fault, NE and OSS alarms are not in sync, ...","[hw fault, ne and oss alarms are not in sync, ...",[Affected services: FAN. Host: ManagedElement=...,5,4.226087,85,...,0.0,0.0,0.0,0.000000,0.000000,0.0,CMI7624L_6NB02,CMI7624L_68T02_S11,2026-06-23 08:00:00,65.556


In [310]:
# cell_pd_filtered = cell_pd[
#     cell_pd["SITEID"] == "CMI7624"
# ]

# cell_pd_filtered

In [308]:
# Find SITEID + CELLID where NR_Cell_Availability changes over time

fluctuating_cells = (
    model_df_merged
    .groupby(["SITEID", "CELLID"])["NR_Cell_Availability"]
    .nunique()
    .reset_index(name="unique_values")
)

fluctuating_cells = fluctuating_cells[
    fluctuating_cells["unique_values"] > 1
]

print(f"Number of fluctuating SITEID-CELLID pairs: {len(fluctuating_cells)}")

display(fluctuating_cells)

Number of fluctuating SITEID-CELLID pairs: 5


,SITEID,CELLID,unique_values
46,CMI6333,CMI7673L_6MM04_R11,17
54,CMI7624,CMI7624L_68T02_S11,4
55,CMI7624,CMI7624L_68T02_S12,4
154,PCB0168,PCB0168H_7NB01_S01,3
214,SMK0474,SMK0474H_7NB01_S01,2


In [305]:
# site_id = "CMI6333"
# cell_id = "CMI7673L_6MM04_R11"

# values = (
#     cell_pd[
#         (cell_pd["SITEID"] == site_id) &
#         (cell_pd["CELLID"] == cell_id)
#     ]["NR_Cell_Availability"]
#     .dropna()
#     .sort_values()
#     .unique()
# )

# print(f"SITEID : {site_id}")
# print(f"CELLID : {cell_id}")
# print(f"Unique values ({len(values)}):")
# print(values)

In [301]:
model_df_filtered.columns

Index(['SITEID', 'Technology', 'EVENT_START_TIME', 'INCIDENT_NUMBER',
       'alarm_sequence', 'unique_description', 'alarm_summary', 'max_severity',
       'mean_severity', 'critical_alarm_count', 'major_alarm_count',
       'severity_escalation', 'service_affecting_count',
       'service_affecting_ratio', 'alarm_duration_span_hours',
       'avg_alarm_duration_hours', 'max_alarm_duration_hours',
       'uncleared_alarm_count', 'cleared_alarm_ratio', 'tt_linked_alarm_count',
       'backhaul_type', 'alarm_recency_hours', 'alarm_count',
       'unique_alarm_count', 'alarm_diversity', 'top_alarm', 'top_alarm_freq',
       'top_alarm_ratio', 'has_infra_alarm', 'has_hardware_alarm',
       'has_txn_alarm', 'has_battery_alarm', 'has_power_alarm',
       'has_gps_alarm', 'has_ran_alarm', 'category_count',
       'repeat_alarm_count', 'alarm_burst_rate',
       'carrier_resource_allocation_failure',
       'critical_temperature_taken_out_of_service', 'ethernet_link_failure',
       'externa

In [ ]:
######making the plot for the feature selection###

In [349]:
model_df_filtered['CELLID'].value_counts()

CMI7624L_68T02_S11    28
Name: CELLID, dtype: int64

In [326]:
# model_df_filtered = (
#     model_df_filtered
#     .sort_values("COLLECTTIME_TS")
#     .groupby("COLLECTTIME_TS", as_index=False)
#     .mean(numeric_only=True)
#     .reset_index(drop=True)
# )


In [323]:
cell_id='CMI7624L_68T02_S11'
selected_site='CMI7624'

In [332]:
import plotly.graph_objects as go
import plotly.io as pio

figures = []

model_df_filtered = model_df_filtered.sort_values("COLLECTTIME_TS").reset_index(drop=True)

kpis_to_plot1 = [
    "hw_fault",
    "critical_temperature_taken_out_of_service",
    "temperature_high",
    "vswr_over_threshold",
    "no_connection",
    "top_alarm_freq",
    "link_failure",
    "alarm_diversity",
    "has_hardware_alarm",
    "critical_alarm_count",
    "alarm_count",
    "category_count",
    "alarm_duration_span_hours",
    "repeat_alarm_count",
    "max_alarm_duration_hours",
    "ne_and_oss_alarms_are_not_in_sync",
    "xnc_link_to_gnodeb_failure",
    "unique_alarm_count",
    "external_link_failure",
    "has_infra_alarm",
    "top_alarm_ratio",
    "has_ran_alarm",
    "mean_severity",
    "avg_alarm_duration_hours",
    "external_link_to_gnodeb_failure",
    "has_txn_alarm",
    "service_affecting_ratio",
    "carrier_resource_allocation_failure",
    "external_alarm_main_ac_power_failure_external_alarm",
]

kpis_to_plot1 = [k for k in kpis_to_plot1 if k in model_df_filtered.columns]

for kpi in kpis_to_plot1:

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=model_df_filtered["COLLECTTIME_TS"],
            y=model_df_filtered[kpi],
            mode="lines+markers",
            name=kpi
        )
    )

    fig.add_trace(
        go.Scatter(
            x=model_df_filtered["COLLECTTIME_TS"],
            y=model_df_filtered[target],
            mode="lines+markers",
            name=target,
            yaxis="y2"
        )
    )

    fig.add_shape(
        type="line",
        x0=outage_time, x1=outage_time,
        y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(dash="dash", width=2)
    )

    fig.add_annotation(
        x=outage_time, y=1,
        xref="x", yref="paper",
        text="Outage Start",
        showarrow=True, arrowhead=2, yshift=15
    )

    fig.update_layout(
        title=f"{kpi} - {HOURS_BEFORE_OUTAGE} Hours Before Outage",
        xaxis=dict(title="Time"),
        yaxis=dict(title=kpi),
        yaxis2=dict(
            title="NR Cell Availability",
            overlaying="y",
            side="right",
            range=[0, 105]
        ),
        hovermode="x unified",
        height=500
    )

    figures.append(fig)


# ============================================================
# CREATE HTML REPORT
# ============================================================

html_content = f"""
<html>
<head><title>Pre-Outage Alarm Feature Analysis - {cell_id}</title></head>
<body>

<h1>Pre-Outage Alarm Feature Trend Analysis</h1>
<h2>Site: {selected_site}</h2>
<h2>Cell: {cell_id}</h2>
<p>Outage Start: {outage_time}</p>
<p>Analysis Window: {15} hours before outage</p>
<p>Each graph shows alarm feature behaviour leading up to the outage.</p>
<hr>
"""

for i, fig in enumerate(figures):
    html_content += pio.to_html(fig, full_html=False, include_plotlyjs=(i == 0))
    html_content += "<hr>"

html_content += "</body></html>"

file_name = f"Pre_Outage_Alarm_Feature_Analysis_{cell_id}.html"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(html_content)

print("Report created:", file_name)


Report created: Pre_Outage_Alarm_Feature_Analysis_CMI7624L_68T02_S11.html


In [ ]:
#####creating the model

In [140]:
model_df.head()

,SITEID,Technology,EVENT_START_TIME,INCIDENT_NUMBER,alarm_sequence,unique_description,alarm_summary,max_severity,mean_severity,critical_alarm_count,...,hw_fault,input_power_failure,link_failure,ne_and_oss_alarms_are_not_in_sync,no_connection,service_redundancy_lost,sync_time_and_phase_accuracy_too_low,temperature_high,vswr_over_threshold,xnc_link_to_gnodeb_failure
0,ATG0096,5G,2026-06-25 15:59:24+00:00,INC000101860585,[NE and OSS alarms are not in sync],[ne and oss alarms are not in sync],[],5,5.000000,1,...,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.000000
1,AYT0143,5G,2026-06-25 10:45:31+00:00,INC000101855244,[External Link Failure],[external link failure],[X2 link problem to one or several neighbourin...,5,3.500000,1,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
2,AYT0300,5G,2026-07-14 03:40:23+00:00,INC000102216937,"[XnC Link to GNodeB Failure, External Link Fai...","[xnc link to gnodeb failure, external link fai...",[XnC link problem to gNodeBs listed in Additio...,5,3.714286,4,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.670642
3,AYT0425,5G,2026-07-13 19:28:42+00:00,INC000102211200,"[NE and OSS alarms are not in sync, External L...","[ne and oss alarms are not in sync, external l...",[X2 link problem to one or several neighbourin...,5,4.000000,2,...,0.0,0.0,0.0,0.632842,0.0,0.0,0.0,0.0,0.0,0.000000
4,AYT0539,5G,2026-07-01 04:59:14+00:00,INC000101964213,[External Link Failure],[external link failure],[X2 link problem to one or several neighbourin...,5,5.000000,1,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000


In [ ]:
# kpi_1month_5G_df_copy

In [141]:
kpi_df = (
    kpi_1month_5G_df_copy
    .withColumn(
        "COLLECTTIME_BKK",
        F.to_timestamp(
            F.col("COLLECTTIME"),
            "yyyyMMddHHmmss"
        )
    )
    .withColumn(
        "HOUR_BKK",
        F.date_trunc("hour", F.col("COLLECTTIME_BKK"))
    )
)

In [143]:
kpi_df.columns

['COLLECTTIME',
 'CELLID',
 'SITE',
 'NR_SA_Paging_Discard_Rate',
 'NR_DL_Active_UEs',
 'NR_UL_Active_UEs',
 'NR_DL_MAC_Volume_MB',
 'NR_UL_MAC_Volume_MB',
 'NR_DL_Active_UEs_True',
 'NR_UL_Active_UEs_True',
 'NR_DL_RBSym_Util',
 'NR_UL_RBSym_Util',
 'NR_Msg2_Attempt_SR',
 'NR_Msg2_Attempts',
 'NR_RACH_SR',
 'NR_RACH_Att',
 'NR_UL_MAC_Cell_Thp_total_time_Mbps',
 'NR_DL_MAC_Cell_Thp_total_time_Mbps',
 'NR_DL_HARQ_BLER_256QAM',
 'NR_DL_RLC_BLER',
 'NR_Avg_DL_MAC_Thp_Mbps',
 'NR_Avg_UL_MAC_UE_Thp_Mbps',
 'NR_DL_MAC_Cell_Thp_Mbps',
 'NR_UL_MAC_Cell_Thp_Mbps',
 'NR_UL_HARQ_BLER_64_QAM',
 'NR_UL_RLC_BLER',
 'NR_UL_HARQ_BLER',
 'NR_DL_HARQ_BLER',
 'NR_UL_DTX_Rate',
 'NR_DL_DTX_Rate',
 'NR_Cell_Availability',
 'NR_Avg_UL_PUCCH_SINR_dB',
 'NR_Avg_UL_PUSCH_SINR_dB',
 'NR_RANK_Below3_Rate',
 'NR_Avg_CQI',
 'NR_Avg_CQI_64QAM',
 'NR_Avg_CQI_256QAM',
 'NR_Avg_UL_Pathloss_dB',
 'NR_Avg_UL_Power_Headroom_dB',
 'NR_Avg_UL_Pathloss_95percentile_dB',
 'NR_UL_RSSI',
 'NR_Avg_MCS_Table1',
 'NR_Avg_MCS_Tabl

In [150]:
kpi_df.select("SITE").show(10, truncate=False)

+--------------+
|SITE          |
+--------------+
|ATG0096T_2NB01|
|ATG0096T_2NB01|
|ATG0096T_2NB01|
|AYT0143T_2NB01|
|AYT0143T_2NB01|
|AYT0300T_2NB01|
|AYT0347L_6NB03|
|AYT0347L_6NB03|
|AYT0405T_2NB01|
|AYT0425T_2NB01|
+--------------+
only showing top 10 rows



In [151]:
from pyspark.sql import functions as F

kpi_df = kpi_df.withColumn(
    "SITEID",
    F.substring(F.col("SITE"), 1, 7)
)

In [153]:
sum_kpis = [
    "NR_DL_MAC_Volume_MB", "NR_UL_MAC_Volume_MB",
    "NR_Msg2_Attempts", "NR_RACH_Att",
    "NR_ENDC_Setup_Att", "NR_SA_RRC_ATT",
    "NR_Cell_Availability_Num",
    "DL_Thp_Bad_Sessions", "DL_Thp_Fair_Sessions",
    "DL_Thp_Good_Sessions", "DL_Thp_Excellent_Sessions",
    "UL_Thp_Bad_Sessions", "UL_Thp_Fair_Sessions",
    "UL_Thp_Good_Sessions", "UL_Thp_Excellent_Sessions",
    "DL_Total_Sessions", "UL_Total_Sessions",
]
 
avg_kpis = [
    "NR_SA_Paging_Discard_Rate", "NR_DL_Active_UEs", "NR_UL_Active_UEs",
    "NR_DL_Active_UEs_True", "NR_UL_Active_UEs_True",
    "NR_DL_RBSym_Util", "NR_UL_RBSym_Util",
    "NR_Msg2_Attempt_SR", "NR_RACH_SR",
    "NR_UL_MAC_Cell_Thp_total_time_Mbps", "NR_DL_MAC_Cell_Thp_total_time_Mbps",
    "NR_DL_HARQ_BLER_256QAM", "NR_DL_RLC_BLER",
    "NR_Avg_DL_MAC_Thp_Mbps", "NR_Avg_UL_MAC_UE_Thp_Mbps",
    "NR_DL_MAC_Cell_Thp_Mbps", "NR_UL_MAC_Cell_Thp_Mbps",
    "NR_UL_HARQ_BLER_64_QAM", "NR_UL_RLC_BLER",
    "NR_UL_HARQ_BLER", "NR_DL_HARQ_BLER",
    "NR_UL_DTX_Rate", "NR_DL_DTX_Rate",
    "NR_Cell_Availability",
    "NR_Avg_UL_PUCCH_SINR_dB", "NR_Avg_UL_PUSCH_SINR_dB",
    "NR_RANK_Below3_Rate", "NR_Avg_CQI", "NR_Avg_CQI_64QAM", "NR_Avg_CQI_256QAM",
    "NR_Avg_UL_Pathloss_dB", "NR_Avg_UL_Power_Headroom_dB",
    "NR_Avg_UL_Pathloss_95percentile_dB", "NR_UL_RSSI",
    "NR_Avg_MCS_Table1", "NR_Avg_MCS_Table2", "NR_TA",
    "NR_Avg_UL_MAC_Thp_EBS_Mbps", "NR_Avg_DL_MAC_Thp_EBS_Mbps",
    "NR_Avg_DL_Latency_ms",
    "NR_ENDC_Setup_SR", "NR_SA_RRC_SSSR", "NR_SA_NG_Sig_SR",
    "NR_ENDC_Inter_sgNodeB_PSCell_Change_SR", "NR_ENDC_Intra_sgNodeB_PSCell_Change_SR",
    "NR_SA_NR_to_LTE_RWR_Ratio",
    "NR_SA_INTER_GNB_HO_SR", "NR_SA_INTRA_GNB_HO_SR",
    "NR_SA_HO_Prep_Succ_Rate", "NR_SA_HO_Exec_Succ_Rate", "NR_SA_HO_SR",
    "NR_ENDC_SARR", "NR_ENDC_RRC_CU_Max", "NR_SA_RRC_CU_Max", "NR_ENDC_RRC_CU_Avg",
    "NR_SA_DRB_EST_SR", "NR_SA_Drop_Rate",
    "NR_SgNB_Abnormal_RR", "NR_RRC_SSR",
    "ESS_TOTAL_PRB_AVG_UTIL_DL", "ESS_NR_PRB_AVG_UTIL_DL",
]
 
agg_exprs = (
    [F.sum(c).alias(c) for c in sum_kpis] +
    [F.avg(c).alias(c) for c in avg_kpis]
)
 
hourly_df = (
    kpi_df
    .groupBy("SITEID", "CELLID", "HOUR_BKK")
    .agg(*agg_exprs)
    .withColumn(
        "COLLECTTIME",
        F.date_format(F.col("HOUR_BKK"), "yyyyMMddHHmmss")
    )
    .drop("HOUR_BKK")
    .orderBy("SITEID", "CELLID", "COLLECTTIME")
)

In [154]:
hourly_df.columns

['SITEID',
 'CELLID',
 'NR_DL_MAC_Volume_MB',
 'NR_UL_MAC_Volume_MB',
 'NR_Msg2_Attempts',
 'NR_RACH_Att',
 'NR_ENDC_Setup_Att',
 'NR_SA_RRC_ATT',
 'NR_Cell_Availability_Num',
 'DL_Thp_Bad_Sessions',
 'DL_Thp_Fair_Sessions',
 'DL_Thp_Good_Sessions',
 'DL_Thp_Excellent_Sessions',
 'UL_Thp_Bad_Sessions',
 'UL_Thp_Fair_Sessions',
 'UL_Thp_Good_Sessions',
 'UL_Thp_Excellent_Sessions',
 'DL_Total_Sessions',
 'UL_Total_Sessions',
 'NR_SA_Paging_Discard_Rate',
 'NR_DL_Active_UEs',
 'NR_UL_Active_UEs',
 'NR_DL_Active_UEs_True',
 'NR_UL_Active_UEs_True',
 'NR_DL_RBSym_Util',
 'NR_UL_RBSym_Util',
 'NR_Msg2_Attempt_SR',
 'NR_RACH_SR',
 'NR_UL_MAC_Cell_Thp_total_time_Mbps',
 'NR_DL_MAC_Cell_Thp_total_time_Mbps',
 'NR_DL_HARQ_BLER_256QAM',
 'NR_DL_RLC_BLER',
 'NR_Avg_DL_MAC_Thp_Mbps',
 'NR_Avg_UL_MAC_UE_Thp_Mbps',
 'NR_DL_MAC_Cell_Thp_Mbps',
 'NR_UL_MAC_Cell_Thp_Mbps',
 'NR_UL_HARQ_BLER_64_QAM',
 'NR_UL_RLC_BLER',
 'NR_UL_HARQ_BLER',
 'NR_DL_HARQ_BLER',
 'NR_UL_DTX_Rate',
 'NR_DL_DTX_Rate',
 'NR_Ce

AnalysisException: Column 'HOUR_BKK' does not exist. Did you mean one of the following? [NR_TA, CELLID, NR_RRC_SSR, NR_UL_RSSI, SITEID, NR_Avg_CQI, NR_RACH_Att, NR_RACH_SR, COLLECTTIME, NR_SA_HO_SR, NR_UL_RLC_BLER, NR_DL_RLC_BLER, NR_ENDC_SARR, NR_SA_RRC_ATT, NR_UL_HARQ_BLER, NR_DL_HARQ_BLER, NR_SA_RRC_SSSR, NR_UL_DTX_Rate, NR_DL_DTX_Rate, NR_UL_RBSym_Util, NR_DL_RBSym_Util, NR_SA_DRB_EST_SR, NR_SA_Drop_Rate, NR_SA_NG_Sig_SR, NR_SA_RRC_CU_Max, NR_UL_Active_UEs, NR_Avg_CQI_64QAM, NR_DL_Active_UEs, NR_ENDC_Setup_SR, NR_Msg2_Attempts, DL_Total_Sessions, NR_Avg_CQI_256QAM, NR_Avg_MCS_Table1, NR_Avg_MCS_Table2, NR_ENDC_RRC_CU_Avg, NR_ENDC_RRC_CU_Max, NR_ENDC_Setup_Att, NR_RANK_Below3_Rate, UL_Total_Sessions, DL_Thp_Bad_Sessions, NR_Msg2_Attempt_SR, NR_UL_MAC_Volume_MB, UL_Thp_Bad_Sessions, NR_DL_MAC_Volume_MB, NR_SA_INTER_GNB_HO_SR, NR_SA_INTRA_GNB_HO_SR, NR_SgNB_Abnormal_RR, NR_UL_HARQ_BLER_64_QAM, DL_Thp_Fair_Sessions, DL_Thp_Good_Sessions, ESS_NR_PRB_AVG_UTIL_DL, NR_Avg_DL_Latency_ms, NR

In [ ]:
####Alarm eda (by merging the KPI and the alarm data)

NameError: name 'spark' is not defined